# Free recall: an optimized neural network discovers multiple memory strategies

> **Li, M., Jensen, K. T., Zhang, Q., Lu, Q. & Mattar, M. G. (2026). *[A neural network model of free recall learns multiple memory strategies.](https://doi.org/10.1038/s42256-026-01274-0)* Nature Machine Intelligence 8, 1238–1250** .

---

Humans recalling a freshly studied list show strikingly structured behaviour: they tend to
recall items in the order they were studied (*temporal contiguity effect*), and memory experts
can recall near-perfectly by mentally walking through a *memory palace*. Classic cognitive
models — the **temporal context model (TCM)** family — explain the first pattern by letting a
slowly drifting "context" state bind items together. But TCM cannot express the second
strategy, because a memory palace represents each item's **serial position independent of its
content**. This notebook reproduces the paper's central demonstration: A recurrent neural network with an episodic memory module, trained **only** to maximize the number of correctly recalled items, spontaneously discovers several qualitatively different retrieval strategies — TCM-like drifting-context strategies **and** a memory-palace-like strategy based on an item-invariant **index code**.

Following the paper, we train a **population of 14 networks** under three training conditions (plus a reservoir control), cluster them into the three strategies exactly as the paper does (k-means over behaviour and representation metrics), and then produce every signature figure **per strategy**: conditional-recall-probability curves, hidden-state PCA trajectories, variance explained by item index vs identity, cross-phase decoding, and noise robustness.

## Setup

In [ ]:
import math
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.stats import rankdata
from sklearn.linear_model import LinearRegression, RidgeClassifier
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

import neuralrnn  # registers the NeuralRNN families
from neuralrnn import AutoModel
from neuralrnn.models.actor_critic import ActorCriticConfig, ActorCriticModel
from neuralrnn.models.gated_rnn import GatedRNNConfig, GatedRNNModel
from neuralrnn.configuration_utils import NeuralRNNConfig
from neuralrnn.modeling_utils import NeuralDynamicsModel
from neuralrnn.auto.modeling_auto import register_model
from neuralrnn.rl.envs.base import SyncVectorEnv
from neuralrnn.analysis import fit_pca

# ---------------------------------------------------------------- paths
# models and figures live next to this notebook (load-first protocol)
NOTEBOOK_DIR = Path.cwd()
MODEL_DIR = NOTEBOOK_DIR / "models" / "free_recall"
POP_DIR = MODEL_DIR / "population"
FIG_DIR = NOTEBOOK_DIR / "figs" / "free_recall"

# training uses the GPU (batch-64 vectorized rollouts, see Part IV);
# recording/analysis run on the same device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

# task hyperparameters (paper Methods: "Free recall")
NUM_FEATURES = 6        # each item = 6 binary features  -> 2**6 = 64 unique items
FEATURE_DIM = 2
SEQUENCE_LEN = 8        # items per list (and recall steps per trial)
VOCAB_SIZE = FEATURE_DIM ** NUM_FEATURES
ACTION_DIM = VOCAB_SIZE + 1     # 64 item actions + 1 "no action"
CORRECT_REWARD = 1.0
WRONG_REWARD = -0.75

def set_seed(seed=SEED):
    """Seed every source of randomness used in the notebook."""
    torch.manual_seed(seed)
    np.random.seed(seed)

set_seed()
print(f"device: {DEVICE}")

---
## Background

### The free-recall paradigm

In **immediate free recall** a participant studies a short list of items (here: 8 words drawn
from a 64-item dictionary) and then recalls them **in any order**. Decades of experiments
(Kahana 1996; Howard & Kahana 1999; Healey et al. 2019) established two robust behavioural signatures:

* **Serial position effects** — enhanced recall of the first items (*primacy*) and last items
  (*recency*) of the list.
* **Temporal contiguity effect** — conditional on having just recalled the item at serial
  position *i*, the next recall is very likely to come from a **nearby position** *i + lag*,
  with a bias toward **forward** transitions (lag > 0). The standard visualization is the
  **conditional-recall-probability (CRP) curve**: the probability of a transition as a
  function of its lag.

### Temporal context models (TCM)

The dominant mechanistic explanation (Howard & Kahana 2002; Sederberg et al. 2008; Polyn et
al. 2009) posits an internal **context** state $\mathbf{c}_t$ that drifts smoothly as items
are experienced. Studying item $i$ with item-vector $\mathbf{e}_i$ updates

$$\mathbf{c}^{(pre)}_{i+1} = \rho\, \mathbf{c}^{(pre)}_i + \beta\, \mathbf{e}_i ,$$

and the item is associated with the *pre* context through an associative matrix
$\mathbf{S}$. During recall, the currently active context $\mathbf{c}$ cues
$\mathbf{S}\mathbf{c}$, the retrieved item is re-integrated (recall reinstates its
studying context), and the new context cues the next item. Because the cue keeps drifting
**toward the items just recalled**, TCM naturally produces a smooth CRP peak near lag 0 and a
forward bias — it is, in effect, a diffusion along the studied list.

### The memory palace (method of loci)

Memory experts use a qualitatively different device: they "place" each item at a
**pre-memorized location** (a stroll through a familiar house) and retrieve by **retracing
the locations in a fixed order**. Computationally this is a **stable, item-independent
scaffold**: the retrieval cue for item $k$ encodes its *serial position* $k$, not its
*content* or a drifting context. Positional coding models capture serial-position data, but a
normative account of *why* forward-ordered recall is so effective — and whether one
learning mechanism can produce both TCM-like and palace-like strategies — was missing.

### This paper

The authors train a GRU **context module** + slot-based **episodic memory module** agent with
advantage actor–critic (A2C) on the free-recall task, with **no instruction about recall
order**. Across populations of networks:

1. **Multiple strategies emerge** (Fig. 2). K-means over behaviour (forward asymmetry,
   temporal organization) and representations (index- vs identity-coding) separates three
   clusters: a **memory-palace** strategy (near-perfect forward recall, stereotyped
   trajectories, index-dominant representations), a **TCM-like forward** and a **TCM-like
   backward** strategy (smooth contiguity, identity-dominant representations). A **reservoir**
   control (frozen recurrent weights) defaults to TCM-like behaviour.
2. **The palace is index coding** (Fig. 3): item *index* explains far more hidden-state
   variance than item *identity*; an index decoder trained in the study phase generalizes
   across items and phases; the strength of index coding correlates with forward recall
   across the whole model population.
3. **Training conditions select the strategy** (Fig. 3f,g): high discount factor
   $\gamma$ (whole-list optimization) and a full working-memory **flush** between study and
   recall (which kills the recency cue) favour the palace.
4. **The palace is optimal and robust** (Fig. 4): it outperforms both TCM-like strategies and
   an analytically optimal TCM, tolerates much more neural noise, and is discovered more
   often when trained under noise.
5. **Optimality is task-dependent** (Figs 5, 6): external drifting context or semantic item
   structure pulls models toward TCM-like coding; a *conditional* free-recall task (recall
   only items matching a cue) — which requires content-based retrieval — **never** produces
   the palace.

Here we reproduce points 1–4 (Fig. 2, Fig. 3a–g, Fig. 4a,b) with a reduced population:
4 seeds per condition instead of 20 — see the deviations table in Part X.

---
## The free-recall task

### Trial structure and rewards

Each trial presents a list of $L = 8$ items sampled **without replacement** from
$V = 64$ items ($2^6$: every item is a 6-digit binary feature tuple; its integer code is the
small-endian encoding $\mathrm{item} = \sum_k f_k\, 2^k$). A trial is one episode of
$2L = 16$ model steps:

| steps | phase | input | output / reward |
|---|---|---|---|
| $0 \dots L-1$ | **study** | one-hot item $x_t$ | item $h_t$ is stored into memory slot $t$; reward 0 |
| $L$ | **transition** | all-zero item block | working memory is flushed (`reset_state`); reward 0 |
| $L \dots 2L-1$ | **response** | all-zero item block | one recalled item per step: **+1** if correct and new, **−0.75** if an intrusion / repeat / "no action" |

### Observation (144-dim) and action (65 choices)

The network's input concatenates the task observation with a **meta-learning feedback** of
the previous action and reward (Wang et al. 2019), which is what lets the network know which
items it has already recalled:

```
[ 0: 64)   one-hot of the current item (all zeros from the transition step on)
[64: 78)   condition block (6 feature + 2 value + 6 padding) — always 0 in free recall
[78:143)   one-hot of the previous action  (0 = "no action", a ≥ 1 = item a−1)
[143]      previous reward
```

The action space is `Discrete(65)`: recall item $a-1$, or output "no action" ($a=0$).

The flush between the phases simulates the **distractor task** of human experiments and
forces the network to rely on the episodic memory module rather than on residual working
memory.

In [ ]:
"""Free-recall task environment (Li et al., Nat. Mach. Intell. 2026).

Faithful port of ``tasks/base.py`` + ``tasks/FreeRecall.py`` + the
``MetaLearningEnv`` wrapper of the reference project, folded into a single
gymnasium-style env so that it plugs directly into ``neuralrnn.rl.SyncVectorEnv``.

Episode structure (one trial = 16 model steps, all envs stay in lockstep):

  * study phase, steps 0-7   : one 64-dim one-hot item per step, reward 0.
  * transition,   step  8    : the observation carries no item (the model is
                               told to flush its working memory via
                               ``info["reset_state"]``); reward 0.
  * response phase, steps 8-15: the model outputs one recalled item per step;
                               +1 per correct new item, -0.75 per intrusion /
                               repeat / no-action.

Observation (144-dim), identical layout to the reference input_dim=144:

  [ 0:64]   one-hot item (all zero from the transition step onward)
  [64:78]   condition block (feature 6 + value 2 + padding 6) — always zero in
            free recall (only the conditional-free-recall variant uses it)
  [78:143]  one-hot of the previous action (0 = "no action", a >= 1 = item a-1)
  [143]     previous reward

Action space: ``Discrete(65)`` (64 items + 1 "no action").
"""
import itertools

import numpy as np
from gymnasium import Env, spaces


class FreeRecallEnv(Env):
    """One trial of immediate free recall with meta-learning observation feedback."""

    metadata = {"render_modes": []}

    def __init__(self,
                 num_features: int = 6,
                 feature_dim: int = 2,
                 sequence_len: int = 8,
                 correct_reward: float = 1.0,
                 wrong_reward: float = -0.75,
                 no_action_reward: float = -0.75,
                 include_extra_observation: bool = False,
                 extra_observation_dim: int = 0,
                 seed: int | None = None):
        super().__init__()
        self.num_features = num_features
        self.feature_dim = feature_dim
        self.vocabulary_size = feature_dim ** num_features          # 64 items
        self.sequence_len = sequence_len                            # 8 items / list
        self.retrieve_time_limit = sequence_len                     # 8 recall steps

        self.correct_reward = correct_reward
        self.wrong_reward = wrong_reward
        self.no_action_reward = no_action_reward

        self.include_extra_observation = include_extra_observation
        self.extra_observation_dim = extra_observation_dim if include_extra_observation else 0

        # base observation: [item | extra | cond-feature | cond-value | pad]
        # 64 + 0 + 6 + 2 + 6 = 78 in the default setup (the trailing 6 dims are
        # unused padding kept for exact weight-shape parity with the reference).
        self.base_obs_dim = (self.vocabulary_size + self.extra_observation_dim
                             + num_features + feature_dim + num_features)
        # full observation folds in the meta-learning feedback:
        # one-hot previous action (vocab+1) + previous reward (1)
        self.action_dim = self.vocabulary_size + 1                  # 65
        self.obs_dim = self.base_obs_dim + self.action_dim + 1      # 144

        self.observation_space = spaces.Box(low=-np.inf, high=np.inf,
                                            shape=(self.obs_dim,), dtype=np.float32)
        self.action_space = spaces.Discrete(self.action_dim)

        # all possible items as feature tuples; integer code = sum(f_k * feature_dim**k)
        self.all_stimuli = np.array(list(itertools.product(range(feature_dim),
                                                           repeat=num_features)))
        if seed is not None:
            self.reset(seed=seed)

    # ------------------------------------------------------------------ helpers
    def _convert_item_to_int(self, item):
        """Feature tuple(s) -> integer code (small-endian). (…, num_features) -> (…,)"""
        item = np.asarray(item)
        weights = self.feature_dim ** np.arange(self.num_features)
        return (item * weights).sum(axis=-1).astype(int)

    def _base_obs(self, item_int=None):
        obs = np.zeros(self.base_obs_dim, dtype=np.float32)
        if item_int is not None:
            obs[item_int] = 1.0
        # dims [vocab : vocab+num_features] would carry the condition feature and
        # [.. +feature_dim] the condition value; free recall has no condition, so
        # this block stays zero (reference: condition_feature = condition_value = None).
        return obs

    def _wrapped_obs(self, base_obs):
        obs = np.zeros(self.obs_dim, dtype=np.float32)
        obs[:self.base_obs_dim] = base_obs
        obs[self.base_obs_dim:self.base_obs_dim + self.action_dim] = np.eye(
            self.action_dim, dtype=np.float32)[self.prev_action]
        obs[-1] = self.prev_reward
        return obs

    def _new_random_prev_action(self):
        self.prev_action = int(self.np_random.integers(self.action_dim))
        self.prev_reward = 0.0

    # ------------------------------------------------------------------ gym API
    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if options is not None and "memory_sequence_index" in options:
            # forced item list (used by the cross-decoding record of the paper)
            self.memory_sequence_index = np.asarray(options["memory_sequence_index"])
        else:
            self.memory_sequence_index = self.np_random.choice(
                self.vocabulary_size, self.sequence_len, replace=False)
        self.memory_sequence_int = np.asarray(self.memory_sequence_index, dtype=int)

        self.retrieved = np.zeros(self.vocabulary_size, dtype=bool)
        self.phase = "encoding"
        self.timestep = 0
        self._new_random_prev_action()

        obs = self._wrapped_obs(self._base_obs(int(self.memory_sequence_int[0])))
        info = {"phase": "encoding", "reset_state": False,
                "memory_sequence_int": self.memory_sequence_int.copy()}
        return obs, info

    def step(self, action):
        action = int(np.asarray(action).item()) if np.asarray(action).ndim else int(action)
        self.timestep += 1
        info = {"phase": self.phase, "reset_state": False,
                "correct": 0, "wrong": 0, "not_know": 0, "item": -1}

        if self.phase == "encoding":
            # actions are ignored while studying; reward is always 0
            reward = 0.0
            terminated = False
            if self.timestep >= self.sequence_len:
                # switch to the response phase; the next observation carries no
                # item and asks the agent to flush its working memory
                self.phase = "recall"
                self.timestep = 0
                info["phase"] = "recall"
                info["reset_state"] = True
                obs = self._wrapped_obs(self._base_obs(None))
            else:
                obs = self._wrapped_obs(
                    self._base_obs(int(self.memory_sequence_int[self.timestep])))
        else:
            # response phase: score the recalled item
            if action == 0:
                reward = self.no_action_reward
                info["not_know"] = 1
            else:
                item = action - 1
                info["item"] = item
                if not self.retrieved[item] and np.any(self.memory_sequence_int == item):
                    reward = self.correct_reward
                    info["correct"] = 1
                    self.retrieved[item] = True
                else:
                    reward = self.wrong_reward
                    info["wrong"] = 1
            obs = self._wrapped_obs(self._base_obs(None))
            terminated = self.timestep >= self.retrieve_time_limit
            info["done"] = terminated

        self.prev_action = action
        self.prev_reward = reward
        return obs, float(reward), bool(terminated), False, info


def make_free_recall_env(seed=0, **kwargs):
    """Thunk factory for ``SyncVectorEnv``."""
    def _thunk():
        return FreeRecallEnv(**kwargs)
    return _thunk

In [ ]:
# sanity check: walk one episode and print the timeline
env = make_free_recall_env(seed=1)()
obs, info = env.reset(seed=1)

print(f"observation dim = {env.observation_space.shape[0]}, "
      f"action dim = {env.action_space.n}")
print(f"study list (item codes): {env.memory_sequence_int.tolist()}")
print("t | phase    | reward | notes")
for t in range(16):
    has_item = obs[:64].sum() == 1
    action = 0                                        # dummy policy: "no action"
    obs, r, term, trunc, info = env.step(action)
    note = ("item presented" if has_item else
            ("FLUSH signal" if info["phase"] == "recall" and t < 8 else ""))
    print(f"{t:2d} | {info['phase']:8s} | {r:+.2f}   | {note}")

---
## The model: a gated_rnn GRU context module + slot-based episodic memory

The agent couples two modules (paper Fig. 1b,c):

* a **context module** — $M = 128$ GRU units holding the working-memory state
  $\mathbf{h}_t \in \mathbb{R}^{128}$;
* a **memory module** — $K = L = 8$ slots $\{\mathbf{m}_k \in \mathbb{R}^{128}\}$ that
  store copies of the hidden state ("episodic memories"), cleared at every trial start.

The context module is not a reimplementation: `ValueMemoryGRUCore` **subclasses
NeuralRNN's `GatedRNNModel`** (`gated_rnn` family), so the recurrent weights live in a
standard `nn.GRU` (`weight_ih_l0` / `weight_hh_l0` / biases, gate order
reset–update–new). The only thing `nn.GRU` cannot express is the episodic-memory input,
so the single-step update is written out using the GRU's own parameters, with the
retrieved memory added to the candidate pre-activation.

### Study phase (memory *writing*)

At each study step the GRU consumes the 144-dim observation and its updated state is
appended to the next free slot:

$$\mathbf{h}_t = \phi_\theta(\mathbf{x}_t, \mathbf{h}_{t-1}),
\qquad \mathbf{m}_t \leftarrow \mathbf{h}_t .$$

### GRU with memory input (Eq. 6–9 of the paper)

The cell is a standard GRU **except** that the retrieved episodic memory enters the candidate
state additively:

$$
\begin{aligned}
\mathbf{r}_t &= \sigma\!\left(\mathbf{W}_r \mathbf{x}_t + \mathbf{U}_r \mathbf{h}_{t-1} + \mathbf{b}_r\right) && \text{(reset gate)}\\
\mathbf{z}_t &= \sigma\!\left(\mathbf{W}_z \mathbf{x}_t + \mathbf{U}_z \mathbf{h}_{t-1} + \mathbf{b}_z\right) && \text{(update gate)}\\
\hat{\mathbf{h}}_t &= \tanh\!\left(\mathbf{W}_h \mathbf{x}_t + \mathbf{U}_h (\mathbf{r}_t \odot \mathbf{h}_{t-1}) + \hat{\mathbf{m}}_t + \mathbf{b}_h\right) && \text{(candidate)}\\
\mathbf{h}_t &= (1 - \mathbf{z}_t) \odot \hat{\mathbf{h}}_t + \mathbf{z}_t \odot \mathbf{h}_{t-1} && \text{(blend)}
\end{aligned}
$$

Two variance-preserving noise processes are applied (Eq. 11), both scaled by the current
state's standard deviation:

* **per-step working-memory noise** $p = 0.04$:
  $\mathbf{h}_t \leftarrow \sqrt{1-p}\,\mathbf{h}_t + \sqrt{p}\,\boldsymbol{\epsilon}\,\mathrm{std}(\mathbf{h}_t)$;
* **the flush between phases**: the response phase starts from
  $\mathbf{h}_0 = \sqrt{1-\beta^2}\,\mathbf{h}_{L-1} + \beta\,\boldsymbol{\epsilon}\,\mathrm{std}(\mathbf{h}_{L-1})$, then $\tanh$,
  with $\beta = 1$ by default — the hidden state is *completely* replaced by noise, so
  nothing but the episodic memory module can support recall.

### Response phase (memory *retrieval*)

At the start of each response step the network uses its current state as a retrieval **cue**:
cosine similarity against all stored memories (Eq. 3–5),

$$
s^i_t = \cos(\mathbf{h}_{t-1}, \mathbf{m}_i),
\qquad
\hat{s}^i_t = \frac{e^{s^i_t / \tau}}{\sum_j e^{s^j_t / \tau}},
\qquad
\hat{\mathbf{m}}_t = \sum_i \hat{s}^i_t\, \mathbf{m}_i .
$$

* The softmax temperature starts at $\tau = 0.01$ and is annealed $\times\,0.9$ every
  3 000 iterations down to $\tau_{\min} = 5\times10^{-4}$, so training begins with a smooth,
  differentiable soft retrieval and ends with an almost hard single-memory retrieval.
* For **testing / analysis** the retrieval is made fully hard:
  $\hat{s}_t = \mathrm{onehot}(\arg\max_i s^i_t)$ (the paper's `model_for_record`).
* On the **first** response step the network runs one extra GRU update with zero input
  *before* the first retrieval — it may learn an arbitrary retrieval cue instead of reusing
  the last stored item.

### Policy and value heads

Two linear heads read the hidden state: policy logits
$\boldsymbol{\ell}_t = \mathbf{W}_a \mathbf{h}_t \in \mathbb{R}^{65}$ with
$\pi_\theta(a_t) = \mathrm{softmax}(\boldsymbol{\ell}_t)$, and a scalar value
$V_t = \mathbf{W}_v \mathbf{h}_t$. All parameters (core + heads) are initialized
$\sim \mathcal{U}(-128^{-1/2},\,128^{-1/2})$, as in the reference implementation
(PyTorch's `nn.GRU` default init is already exactly this).

In [ ]:
"""Value-memory GRU agent (Li et al., Nat. Mach. Intell. 2026).

Notebook-level NeuralRNN family: the paper's "context module + memory module"
architecture, faithful to ``models/model/value_gru.py`` +
``models/memory/value.py`` + ``models/memory/similarity/similarity.py`` of the
reference project.

The context module IS the gated_rnn family's GRU: :class:`ValueMemoryGRUCore`
subclasses :class:`~neuralrnn.models.gated_rnn.GatedRNNModel`, so the recurrent
weights live in a standard ``nn.GRU`` (``gru.weight_ih_l0`` / ``gru.weight_hh_l0``
/ biases, gate order reset-update-new) and every gated_rnn tool (freeze groups,
AutoConfig / AutoModel serialization, dtype handling) applies unchanged. The
only addition is the episodic memory module: the retrieved memory is injected
into the GRU candidate pre-activation, which ``nn.GRU`` cannot express, so the
single-step update is written out manually using the ``nn.GRU`` parameters
(exactly the equations of ``GatedRNNModel._gru_step`` plus the memory term)::

    r_t = sigmoid(W_ir x_t + b_ir + W_hr h_{t-1} + b_hr)
    z_t = sigmoid(W_iz x_t + b_iz + W_hz h_{t-1} + b_hz)   (update/keep gate)
    n_t = tanh( W_in x_t + b_in + r_t*(W_hn h_{t-1} + b_hn) + g*m_t )
    h_t = (1 - z_t) * n_t + z_t * h_{t-1}

with per-step multiplicative working-memory noise (variance-preserving) and a
variance-preserving "flush" of the hidden state between the study and response
phases followed by ``tanh``. PyTorch's default ``nn.GRU`` init is already the
reference init U(+-1/sqrt(hidden)).

Memory module — ``capacity`` slots holding hidden-state copies::

    study step t    : slot[to_replace] <- h_t            (write)
    response step t : s_t  = cos(h_{t-1}, m_k)           (cue = current state)
                      w_t  = softmax(s_t / tau)          (tau: 0.01 -> 0.0005)
                      mhat = sum_k w_t[k] m_k            ("weight_sum", training)
                           = m_{argmax w_t}              ("argmax", testing)

The first response step additionally runs ``evolve_steps`` GRU updates with
zero input *before* the first retrieval, so the network can build a retrieval
cue that is independent of the last stored item.

The class keeps the ``NeuralDynamicsModel`` contract (recurrence/readout/
init_state), so it plugs into ``ActorCriticModel`` unchanged: policy =
softmax(actor(h)), value = critic(h).

Reference hyperparameters (experiments/Basic/setups/setup.json):
    hidden 128, input 144 (= 78 obs + 64 prev-action one-hot + 1 prev-reward),
    65 actions, capacity 8, tau0 = 0.01 (decay x0.9, min 5e-4),
    flush_noise 1.0, wm_noise_prop 0.04, weight-sum retrieval, policy temp 1.0,
    all weights init U(-1/sqrt(128), 1/sqrt(128)).
"""

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from neuralrnn.modeling_utils import NeuralDynamicsModel
from neuralrnn.auto.modeling_auto import register_model
from neuralrnn.models.gated_rnn import GatedRNNConfig, GatedRNNModel


class ValueMemoryGRUConfig(GatedRNNConfig):
    """Configuration of the value-memory GRU core (a gated_rnn GRU + memory).

    Inherits the gated_rnn fields (``input_dim`` / ``latent_dim`` /
    ``output_dim`` / ``rnn_type="GRU"`` / ``dtype``) and adds the episodic
    memory module and noise settings.

    Args:
        mem_capacity: number of episodic memory slots (= list length, 8).
        mem_value_dim: stored vector size (default = latent_dim).
        mem_init_temperature: initial softmax temperature tau of memory retrieval.
        mem_min_temperature: lower bound of the tau annealing (5e-4).
        mem_decay_rate: multiplicative tau annealing rate (0.9), applied by the
            trainer every ``mem_decay_iter`` iterations.
        recall_method: "weight_sum" (differentiable soft retrieval, training)
            or "argmax" (hard single-memory retrieval, testing/analysis).
        flush_noise: variance fraction of the hidden state replaced by noise at
            the study->response transition (1.0 = full working-memory flush).
        wm_noise_prop: per-step working-memory noise proportion (0.04).
        em_noise_prop: noise proportion added to the retrieved memory (0).
        evolve_state_between_phases: run ``evolve_steps`` zero-input GRU updates
            before the first retrieval of a trial.
        evolve_steps: number of those extra updates (1).
        reservoir: if True, freeze the recurrent weights (gru.weight_hh_l0 and
            gru.bias_hh_l0) so only the input/output weights train — the paper's
            "reservoir RNN" control.
    """

    model_type = "value_memory_gru"

    def __init__(self,
                 input_dim: int = 144,
                 latent_dim: int = 128,
                 output_dim: int | None = None,
                 mem_capacity: int = 8,
                 mem_value_dim: int | None = None,
                 mem_init_temperature: float = 0.01,
                 mem_min_temperature: float = 5e-4,
                 mem_decay_rate: float = 0.9,
                 recall_method: str = "weight_sum",
                 flush_noise: float = 1.0,
                 wm_noise_prop: float = 0.04,
                 em_noise_prop: float = 0.0,
                 evolve_state_between_phases: bool = True,
                 evolve_steps: int = 1,
                 reservoir: bool = False,
                 **kwargs) -> None:
        kwargs.pop("rnn_type", None)   # the memory injection assumes GRU gates
        super().__init__(input_dim=input_dim, latent_dim=latent_dim,
                         output_dim=output_dim or latent_dim, rnn_type="GRU",
                         **kwargs)
        self.mem_capacity = int(mem_capacity)
        self.mem_value_dim = int(mem_value_dim) if mem_value_dim else int(latent_dim)
        self.mem_init_temperature = float(mem_init_temperature)
        self.mem_min_temperature = float(mem_min_temperature)
        self.mem_decay_rate = float(mem_decay_rate)
        if recall_method not in ("weight_sum", "argmax"):
            raise ValueError(f"recall_method must be 'weight_sum' or 'argmax', "
                             f"got {recall_method!r}")
        self.recall_method = recall_method
        self.flush_noise = float(flush_noise)
        self.wm_noise_prop = float(wm_noise_prop)
        self.em_noise_prop = float(em_noise_prop)
        self.evolve_state_between_phases = bool(evolve_state_between_phases)
        self.evolve_steps = int(evolve_steps)
        self.reservoir = bool(reservoir)


@register_model("value_memory_gru")
class ValueMemoryGRUCore(GatedRNNModel):
    """gated_rnn GRU context module + slot-based episodic memory module.

    Trial-phase control (driven by the training / recording loop, exactly like
    the reference ``set_encoding`` / ``set_retrieval`` / ``reset_memory``):

        core.reset_memory(batch)               # clear slots at trial start
        core.set_phase(encoding=True)          # study step: write h_t into a slot
        core.set_phase(retrieving=True)        # response step: retrieve then update
    """

    config_class = ValueMemoryGRUConfig

    def __init__(self, config: ValueMemoryGRUConfig) -> None:
        super().__init__(config)
        # The core's readout is the identity (policy/value heads live on the
        # RL agent), so the gated_rnn readout layer is dropped.
        del self.readout_layer
        if config.reservoir:
            # paper's reservoir RNN: freeze the recurrent transformation,
            # train input weights (gru.weight_ih_l0) and the readout heads
            self.gru.weight_hh_l0.requires_grad = False
            self.gru.bias_hh_l0.requires_grad = False
        # episodic memory buffer: (batch, capacity, value_dim); plain tensors so
        # the in-graph write -> retrieve path keeps gradients within a trial.
        self.register_memory(config.mem_capacity, config.mem_value_dim, batch=1)
        self.stored_memory = 0
        self.to_be_replaced = 0
        self.encoding = False
        self.retrieving = False
        self.last_encoding = False
        self.mem_temperature = config.mem_init_temperature
        self.last_similarity: torch.Tensor | None = None
        self.last_cue: torch.Tensor | None = None

    # -------------------------------------------------------- memory management
    def register_memory(self, capacity: int, value_dim: int, batch: int = 1):
        self.values = torch.zeros(batch, capacity, value_dim)
        return self.values

    def reset_memory(self, batch_size: int | None = None, flush: bool = True) -> None:
        """Clear all slots (called at the start of every trial)."""
        if batch_size is None:
            batch_size = self.values.shape[0]
        if flush or self.values.shape[0] != batch_size:
            device = self.gru.weight_ih_l0.device
            self.values = torch.zeros(batch_size, self.config.mem_capacity,
                                      self.config.mem_value_dim, device=device)
        self.stored_memory = 0
        self.to_be_replaced = 0
        self.encoding = False
        self.retrieving = False
        self.last_encoding = False
        self.last_similarity = None
        self.last_cue = None

    def set_phase(self, encoding: bool = False, retrieving: bool = False) -> None:
        self.encoding, self.retrieving = encoding, retrieving

    def decay_mem_temperature(self) -> bool:
        """tau <- rate * tau once per annealing interval; True if it changed."""
        if self.mem_temperature > self.config.mem_min_temperature:
            self.mem_temperature = max(self.mem_temperature * self.config.mem_decay_rate,
                                       self.config.mem_min_temperature)
            return True
        return False

    # -------------------------------------------------------- NeuralDynamics contract
    def init_state(self, batch_size: int, device: str | torch.device = "cpu",
                   recall: bool = False, prev_state: torch.Tensor | None = None):
        """Trial-start state.

        Encoding phase: zeros (the reference "noise" init for study). Response
        phase: variance-preserving flush of the last study state::

            h0 = sqrt(1 - beta^2) * h_N + beta * eps * std(h_N),   then tanh
        """
        if not recall:
            return torch.zeros(batch_size, self.config.latent_dim,
                               device=device, requires_grad=True)
        beta = self.config.flush_noise
        state = (math.sqrt(max(0.0, 1.0 - beta ** 2)) * prev_state
                 + beta * torch.randn_like(prev_state) * prev_state.std())
        return torch.tanh(state)

    def readout(self, z_t):
        """Identity map (the policy/value heads live on the RL agent)."""
        return z_t

    def recurrence(self, x_t, z_prev, *, inputs=None):
        """One model step: (retrieve ->) GRU update (-> write memory)."""
        h = z_prev
        mem = None
        if self.retrieving and self.config.evolve_state_between_phases and self.last_encoding:
            # one (or evolve_steps) zero-input update(s) to build the first cue
            for _ in range(self.config.evolve_steps):
                h = self._gru_step_mem(torch.zeros_like(x_t), h, None)
            self.last_encoding = False
        if self.retrieving and self.stored_memory > 0:
            self.last_cue = h                     # the state that drove retrieval
            mem, sim = self.retrieve(h)
            self.last_similarity = sim
        else:
            # reference behaviour: the recorded similarity is zeros outside the
            # response phase (its entropy then contributes ~0 to the aux loss)
            self.last_similarity = torch.zeros(
                x_t.shape[0], self.config.mem_capacity, device=x_t.device)
        h = self._gru_step_mem(x_t, h, mem)
        if self.encoding:
            self.encode(h)
        return h

    def forward(self, inputs: torch.Tensor | None = None, **kwargs):
        """Full-sequence rollout via the generic recurrence loop.

        ``GatedRNNModel.forward`` would call ``nn.GRU`` directly, which cannot
        inject the retrieved memory into the candidate pre-activation — so this
        core explicitly falls back to the step-by-step
        ``NeuralDynamicsModel.forward`` (recurrence + readout per step), where
        the memory writes/retrievals happen.
        """
        return NeuralDynamicsModel.forward(self, inputs, **kwargs)

    # -------------------------------------------------------- elementary pieces
    def _gru_step_mem(self, x_t, h, mem: torch.Tensor | None) -> torch.Tensor:
        """``GatedRNNModel._gru_step`` + memory in the candidate pre-activation.

        Uses the standard ``nn.GRU`` parameters (gate order reset, update, new)::
            r = sigmoid(W_ir x + b_ir + W_hr h + b_hr)
            z = sigmoid(W_iz x + b_iz + W_hz h + b_hz)
            n = tanh(W_in x + b_in + r * (W_hn h + b_hn) + g*m)   (g = 1)
            h' = (1 - z) * n + z * h
        followed by the variance-preserving working-memory noise.
        """
        W_ih, W_hh = self.gru.weight_ih_l0, self.gru.weight_hh_l0
        b_ih, b_hh = self.gru.bias_ih_l0, self.gru.bias_hh_l0
        M = h.shape[-1]
        W_ir, W_iz, W_in = W_ih.split(M, dim=0)
        W_hr, W_hz, W_hn = W_hh.split(M, dim=0)
        b_ir, b_iz, b_in = b_ih.split(M, dim=0)
        b_hr, b_hz, b_hn = b_hh.split(M, dim=0)

        r = torch.sigmoid(x_t @ W_ir.t() + b_ir + h @ W_hr.t() + b_hr)
        z = torch.sigmoid(x_t @ W_iz.t() + b_iz + h @ W_hz.t() + b_hz)
        preact = x_t @ W_in.t() + b_in + r * (h @ W_hn.t() + b_hn)
        if mem is not None:
            mem = self._mem_noise(mem)
            preact = preact + mem          # em_gate == 1.0 ("constant" gate)
        n = torch.tanh(preact)
        h = (1 - z) * n + z * h
        p = self.config.wm_noise_prop
        if p > 0:
            h = math.sqrt(1.0 - p) * h + math.sqrt(p) * torch.randn_like(h) * h.std()
        return h

    def _mem_noise(self, mem):
        p = self.config.em_noise_prop
        if p > 0:
            mem = (math.sqrt(1.0 - p) * mem
                   + math.sqrt(p) * torch.randn_like(mem) * mem.std())
        return mem

    def encode(self, state: torch.Tensor) -> None:
        """Write the current hidden state into the next slot."""
        idx = torch.tensor(self.to_be_replaced, device=state.device)
        onehot = F.one_hot(idx, self.config.mem_capacity).to(state.dtype)
        self.values = self.values + torch.einsum("k,bv->bkv", onehot, state)
        self.to_be_replaced = (self.to_be_replaced + 1) % self.config.mem_capacity
        self.stored_memory = min(self.stored_memory + 1, self.config.mem_capacity)
        self.last_encoding = True

    def retrieve(self, query: torch.Tensor):
        """Cosine-similarity retrieval.

        Returns (retrieved_memory, similarity_weights) where the weights are the
        softmaxed similarities (the quantity whose entropy the auxiliary loss
        sharpens, and whose argmax indexes the retrieved slot in analyses).
        """
        sim = F.cosine_similarity(query.unsqueeze(1), self.values.to(query.device), dim=-1)  # (B,K)
        tau = max(self.mem_temperature, 1e-8)
        weights = F.softmax(sim / tau, dim=-1)
        if self.config.recall_method == "argmax":
            weights = F.one_hot(weights.argmax(-1), self.config.mem_capacity).to(query.dtype)
        retrieved = torch.bmm(weights.unsqueeze(1), self.values).squeeze(1)  # (B,V)
        return retrieved, weights


# --------------------------------------------------------------------- factory
def make_free_recall_agent(core_config: ValueMemoryGRUConfig | None = None,
                           action_dim: int = 65,
                           core_weights_from=None,
                           device: str = "cpu"):
    """Build the full RL agent: value-memory GRU core + actor/critic heads.

    ``core_weights_from``: an agent (or core) whose weights are copied in — used
    to turn a trained model into the record model (argmax retrieval, tau at its
    minimum), mirroring the reference ``model_for_record`` protocol.
    """
    from neuralrnn.models.actor_critic import ActorCriticConfig, ActorCriticModel

    if core_config is None:
        core_config = ValueMemoryGRUConfig()
    agent = ActorCriticModel(ActorCriticConfig(
        core_config=core_config.to_dict(),
        action_dim=action_dim,
        action_type="discrete",
        readout_e_only=False,
        value_bias=True,
    ))
    if core_weights_from is not None:
        src = core_weights_from.core if hasattr(core_weights_from, "core") else core_weights_from
        agent.core.load_state_dict(src.state_dict())
    agent.core.reset_memory(1)
    return agent.to(device)


def init_reference_uniform_(agent) -> None:
    """Reference parameter init: every parameter ~ U(-1/sqrt(H), 1/sqrt(H)).

    (The ``nn.GRU`` default init is already exactly this; the call additionally
    puts the actor/critic heads on the same scale, as the reference does.)
    """
    std = 1.0 / math.sqrt(agent.core.config.latent_dim)
    with torch.no_grad():
        for p in agent.parameters():
            if p.requires_grad:
                p.data.uniform_(-std, std)


def make_record_agent(trained_agent, device: str = "cpu"):
    """The paper's ``model_for_record`` protocol: identical weights, but hard
    single-memory retrieval ("argmax") at the minimal temperature tau = 5e-4."""
    cfg = trained_agent.core.config
    record_cfg = ValueMemoryGRUConfig(
        input_dim=cfg.input_dim, latent_dim=cfg.latent_dim,
        mem_capacity=cfg.mem_capacity, mem_value_dim=cfg.mem_value_dim,
        mem_init_temperature=cfg.mem_min_temperature,
        mem_min_temperature=cfg.mem_min_temperature,
        mem_decay_rate=cfg.mem_decay_rate,
        recall_method="argmax",
        flush_noise=cfg.flush_noise, wm_noise_prop=cfg.wm_noise_prop,
        em_noise_prop=cfg.em_noise_prop,
        evolve_state_between_phases=cfg.evolve_state_between_phases,
        evolve_steps=cfg.evolve_steps, reservoir=cfg.reservoir)
    agent = make_free_recall_agent(record_cfg, action_dim=trained_agent.actor.out_features,
                                   core_weights_from=trained_agent, device=device)
    # copy the FULL state (core + actor + critic heads), exactly like the
    # reference model_for_record.load_state_dict(torch.load(model.pt))
    agent.load_state_dict(trained_agent.state_dict())
    return agent

In [ ]:
# ---------------------------------------------------------------- RL agent
# The core plugs into the generic ActorCriticModel shell unchanged. As a sanity
# check, the memory-free step of our core must reduce EXACTLY to the gated_rnn
# GRU step (same parameters, same equations):

set_seed(SEED)
core_config = ValueMemoryGRUConfig(flush_noise=1.0)
agent = make_free_recall_agent(core_config, action_dim=ACTION_DIM)
init_reference_uniform_(agent)
n_params = sum(p.numel() for p in agent.parameters() if p.requires_grad)
print(f"trainable parameters: {n_params:,}")

plain_gru = GatedRNNModel(GatedRNNConfig(input_dim=144, latent_dim=128,
                                         output_dim=128))
plain_gru.load_state_dict(
    {k: v for k, v in agent.core.state_dict().items()
     if k in plain_gru.state_dict()}, strict=False)
clean_core = ValueMemoryGRUCore(ValueMemoryGRUConfig(wm_noise_prop=0.0))
clean_core.load_state_dict(agent.core.state_dict())
x_demo, h_demo = torch.randn(3, 144), torch.randn(3, 128)
assert torch.allclose(clean_core._gru_step_mem(x_demo, h_demo, None),
                      plain_gru._gru_step(x_demo, h_demo))
print("memory-free step == gated_rnn GRU step: OK")

---
## Training: advantage actor–critic, and the population protocol

### A2C on the recall task

The model is trained with **A2C** (Eq. 10 of the paper): one gradient update per batch of
episodes, using only the response-phase outputs (study-phase rewards and outputs carry no
learning signal):

$$
\Delta\theta \;\propto\; \sum_{t=1}^{T}
\Big[ \nabla_\theta \log \pi_\theta(a_t) \, \big(R_t - V_t\big)
+ \tfrac12 \nabla_\theta \big(V_t - R_t\big)^2
+ \eta_e \nabla_\theta H[\pi_\theta(\cdot|\mathbf{h}_t)]
\Big],
$$

with the discounted return $R_t = \sum_k \gamma^{k-t} r_k$ (default $\gamma = 0.999$),
$\eta_e = 0.02$, and log₂-entropies as in the reference.

**Memory-entropy regularizer.** To push retrieval toward *single* memories instead of
blends of many, the paper adds a penalty on the entropy of the retrieval weights
(computed over **all** steps; study-phase weights are zero and contribute ≈ 0):

$$L_\mathrm{mem} = \lambda \sum_t \sum_k H[\hat{s}^k_t],
\qquad \lambda = 10^{-3}.$$

**Optimization**: Adam (lr $3\times10^{-3}$), grad-norm clip 1.0,
`ReduceLROnPlateau` (factor 0.1, patience 30) stepped on $-\langle r\rangle$, $\tau$
annealed every 3 000 iterations; training stops when train accuracy ≥ 0.99 **and**
$\tau$ has reached its floor (or after $10^5$ iterations × 16 episodes, the paper's
budget of 1.6 M episodes).

### The population: which conditions select which strategy

The paper's key causal manipulation (Fig. 3f,g) varies two training hyperparameters:

* **discount factor $\gamma$** — small $\gamma$ myopically rewards the *next* few recalls;
  $\gamma \to 1$ optimizes the **whole list**, favouring recall initiation at the start
  of the list (the palace);
* **working-memory flush $\beta$** — with 0% flush the response phase starts from the last
  study state, so the *most recent items* form the retrieval cue (recency) and models
  prefer **backward** recall; a full flush (100%) removes that bias and the palace
  dominates.

We train 4 seeds per condition (the paper trains 20):

| condition | $\gamma$ | flush $\beta$ | expected strategy (paper Fig. 3f,g) |
|---|---|---|---|
| `g999_f100` | 0.999 | 1.0 | memory palace |
| `g020_f100` | 0.2 | 1.0 | TCM-like forward |
| `g999_f000` | 0.999 | 0.0 | TCM-like backward |
| `reservoir` | 0.999 | 1.0 | (frozen recurrent weights; the paper's no-learning control) |

> **GPU recipe.** The rollout loop is step-by-step (16 steps per trial), so at the paper's
> batch size of 16 the GPU is launch-latency-bound and *slower* than the CPU. We therefore
> train with **batch 64 × 25 000 iterations on the GPU** — the identical per-episode budget
> (1.6 M episodes), with the iteration-based schedules (logging, $\tau$ annealing, lr
> plateau) rescaled by the batch ratio 64/16 so the per-episode protocol is unchanged.

> **Load-first**: the 14 models are pre-trained by `fr_tests/train_population.py`
> (`python fr_tests/train_population.py --all --device cuda`). If the checkpoints exist
> they are simply loaded and this section only plots their training curves.

In [ ]:
"""A2C training loop for the free-recall agent (faithful port of the paper).

Reproduces ``train/train.py`` of the reference project: one gradient update per
episode batch (16 parallel trials x 16 steps), advantage actor-critic on the
response-phase outputs only, policy-entropy bonus, and the auxiliary
"memory one-hot" regularizer that sharpens retrieval::

    L = -sum_t log pi(a_t) * (R_t - V_t)/B + 0.5 * sum_t (V_t - R_t)^2 / B
        - eta * sum_t H[pi_t]/B + lambda * sum_t sum_k H[w_t[k]]/B

with R_t the discounted return (gamma = 0.999), log2-entropies as in the
reference, grad-norm clip 1.0, Adam(lr=3e-3) + ReduceLROnPlateau(factor 0.1,
patience 30) stepped on -mean reward, and tau annealing (x0.9) every
``mem_decay_iter`` iterations. Training stops at train accuracy >= 0.99 with
tau at its minimum (or after ``num_iter`` iterations).
"""

import json
import math
import time
from pathlib import Path

import numpy as np
import torch

from neuralrnn.rl.envs.base import SyncVectorEnv



def _entropy2(p: torch.Tensor) -> torch.Tensor:
    """Row-wise log2 entropy (reference ``models.utils.entropy``)."""
    p = p + 1e-8
    return -(p * torch.log2(p)).sum(dim=-1)


def compute_returns(rewards: np.ndarray, gamma: float) -> np.ndarray:
    """Discounted returns R_t = sum_k gamma^(k-t) r_k. rewards: (T, B) -> (T, B)."""
    T, B = rewards.shape
    R = np.zeros(B)
    returns = np.zeros_like(rewards, dtype=float)
    for t in range(T - 1, -1, -1):
        R = rewards[t] + gamma * R
        returns[t] = R
    return returns


def rollout_episode(agent, envs: SyncVectorEnv, batch_size: int, device: str,
                    collect: bool = True):
    """Run one full trial batch; collect response-phase quantities for the loss.

    Returns dict with recall-phase log-probs/values/entropies/rewards
    (T=8, B), the memory-similarity weights of every step (16, B, K), and the
    per-action correct/wrong counters.
    """
    core = agent.core
    state = core.init_state(batch_size, device)
    core.reset_memory(batch_size)

    obs, infos = envs.reset()
    core.set_phase(encoding=True)
    logps, values, entropies, rewards, mem_sims = [], [], [], [], []
    correct = wrong = not_know = 0
    T = envs.envs[0].sequence_len * 2  # 16 model steps per trial

    for _ in range(T):
        if infos[0].get("reset_state"):
            state = core.init_state(batch_size, device, recall=True, prev_state=state)
        x = torch.as_tensor(np.asarray(obs), dtype=torch.float32, device=device)
        action, logp, entropy, value, state = agent.step(x, state)
        # the reference computes policy entropies with log2 (models.utils.entropy)
        entropy = entropy / math.log(2.0)
        obs, rew, term, trunc, infos = envs.step(action.detach().cpu().numpy())

        phase_recall = infos[0]["phase"] == "recall"
        core.set_phase(encoding=not phase_recall, retrieving=phase_recall)
        rewards.append(np.asarray(rew, dtype=float))
        logps.append(logp)
        values.append(value)
        entropies.append(entropy)
        mem_sims.append(core.last_similarity)
        for info in infos:
            correct += info.get("correct", 0)
            wrong += info.get("wrong", 0)
            not_know += info.get("not_know", 0)

    n_study = T // 2
    out = {
        "logps": torch.stack(logps[n_study:]),              # (8, B)
        "values": torch.stack(values[n_study:]),            # (8, B)
        "entropies": torch.stack(entropies[n_study:]),      # (8, B)
        "rewards": np.stack(rewards[n_study:]),             # (8, B)
        "mem_sims": torch.stack(mem_sims),                  # (16, B, K)
        "correct": correct, "wrong": wrong, "not_know": not_know,
    }
    return out


def train_free_recall(agent, envs: SyncVectorEnv, *,
                      device: str = "cpu",
                      num_iter: int = 100_000,
                      batch_size: int = 16,
                      test_iter: int = 500,
                      gamma: float = 0.999,
                      eta: float = 0.02,
                      value_weight: float = 1.0,
                      mem_reg_weight: float = 0.001,
                      lr: float = 3e-3,
                      lr_decay_factor: float = 0.1,
                      lr_decay_patience: int = 30,
                      min_lr: float = 1e-7,
                      grad_max_norm: float = 1.0,
                      mem_decay_iter: int = 3000,
                      stop_accu: float = 0.99,
                      save_dir: str | Path | None = None,
                      save_every: int = 10_000,
                      log_every: int | None = None,
                      start_iter: int = 0,
                      history: dict | None = None,
                      verbose: bool = True):
    """Train the agent; returns the history dict and writes checkpoints.

    Resume support: pass ``start_iter`` (the last completed iteration) and the
    previously saved ``history`` to continue a run from its latest checkpoint.
    The caller must restore the annealed retrieval temperature on the core;
    the optimizer/scheduler restart fresh (Adam moments are not checkpointed).
    """
    if log_every is None:
        log_every = test_iter
    agent.to(device)
    core = agent.core
    opt = torch.optim.Adam([p for p in agent.parameters() if p.requires_grad], lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=lr_decay_factor,
        patience=lr_decay_patience, threshold=1e-3, min_lr=min_lr)

    save_dir = Path(save_dir) if save_dir is not None else None
    if history is None:
        history = {"iter": [], "accuracy": [], "error": [], "mean_reward": [],
                   "loss": [], "mem_temperature": [], "lr": []}

    running = dict(correct=0, wrong=0, not_know=0, reward=0.0, loss=0.0)
    t0 = time.time()
    stop = False

    for it in range(start_iter + 1, num_iter + 1):
        ep = rollout_episode(agent, envs, batch_size, device)

        # ---------------- A2C loss on the response phase (reference A2CLoss) --
        returns = torch.as_tensor(
            compute_returns(ep["rewards"], gamma), dtype=torch.float32,
            device=device)                                        # (8, B)
        logps, values = ep["logps"], ep["values"]
        entropies, mem_sims = ep["entropies"], ep["mem_sims"]
        B = batch_size

        advantage = returns - values.detach()
        pg_loss = -(logps * advantage).sum() / B
        v_loss = 0.5 * ((values - returns) ** 2).sum() / B
        ent_bonus = entropies.sum() / B
        mem_reg = _entropy2(mem_sims.reshape(-1, mem_sims.shape[-1])).sum() / B

        loss = pg_loss + value_weight * v_loss - eta * ent_bonus + mem_reg_weight * mem_reg

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in agent.parameters() if p.requires_grad], grad_max_norm)
        opt.step()

        running["correct"] += ep["correct"]
        running["wrong"] += ep["wrong"]
        running["not_know"] += ep["not_know"]
        running["reward"] += float(ep["rewards"].sum())
        running["loss"] += float(loss.detach())

        # ------------------------------------------------- periodic evaluation
        if it % test_iter == 0:
            total = running["correct"] + running["wrong"] + running["not_know"]
            acc = running["correct"] / max(total, 1)
            err = running["wrong"] / max(total, 1)
            mean_r = running["reward"] / (test_iter * batch_size)
            mean_loss = running["loss"] / test_iter
            history["iter"].append(it)
            history["accuracy"].append(acc)
            history["error"].append(err)
            history["mean_reward"].append(mean_r)
            history["loss"].append(mean_loss)
            history["mem_temperature"].append(core.mem_temperature)
            history["lr"].append(opt.param_groups[0]["lr"])

            sched.step(-mean_r)
            if verbose:
                print(f"iter {it:>7d} | acc {acc:.3f} | err {err:.3f} | "
                      f"reward {mean_r:+.3f} | loss {mean_loss:.4f} | "
                      f"tau {core.mem_temperature:.5f} | "
                      f"{(time.time() - t0) / max(it - start_iter, 1):.3f}s/it",
                      flush=True)

            # tau annealing every mem_decay_iter iterations (reference protocol)
            if it % mem_decay_iter == 0:
                core.decay_mem_temperature()

            # early stop: near-perfect accuracy with fully annealed tau
            if acc >= stop_accu and core.mem_temperature <= core.config.mem_min_temperature:
                stop = True

            if save_dir is not None:
                save_dir.mkdir(parents=True, exist_ok=True)
                agent.save_pretrained(str(save_dir))
                with open(save_dir / "history.json", "w", encoding="utf-8") as f:
                    json.dump(history, f)
            running = dict(correct=0, wrong=0, not_know=0, reward=0.0, loss=0.0)

        if save_dir is not None and it % save_every == 0 and not (it % test_iter == 0):
            agent.save_pretrained(str(save_dir))
        if stop:
            if verbose:
                print(f"training end at iter {it} "
                      f"(acc {acc:.3f} >= {stop_accu}, tau {core.mem_temperature:.5f})",
                      flush=True)
            break

    return history

In [ ]:
# ------------------------------------------------- load-first: the population
# Training is done by fr_tests/train_population.py (GPU, ~25 min per model).
# Here we only load the finished checkpoints and show how training went.

def load_history(model_dir):
    with open(model_dir / "history.json", encoding="utf-8") as f:
        return json.load(f)


pop_dirs = sorted(d for d in POP_DIR.iterdir()
                  if d.is_dir() and (d / "run_meta.json").exists()
                  and (d / "record_main.npz").exists()) \
    if POP_DIR.exists() else []
print(f"population: {len(pop_dirs)} trained models in {POP_DIR}")
for d in pop_dirs:
    with open(d / "run_meta.json", encoding="utf-8") as f:
        meta = json.load(f)
    hist = load_history(d)
    print(f"  {d.name:24s} gamma={meta['gamma']:<5} flush={meta['flush']:<4} "
          f"reservoir={meta['reservoir']!s:<5} final acc={hist['accuracy'][-1]:.3f} "
          f"({meta['wall_minutes']} min)")

COND_COLORS = {"g999_f100": "#2A9D8F", "g020_f100": "#E76F51",
               "g999_f000": "#E9C46A", "reservoir": "#264653"}

if pop_dirs:
    plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                         "axes.spines.top": False, "axes.spines.right": False})
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 2.8))
    for d in pop_dirs:
        hist = load_history(d)
        cond = d.name.split("__")[0]
        c = COND_COLORS[cond]
        it = np.asarray(hist["iter"]) * 64 / 1000   # x-axis in k-episodes
        axes[0].plot(it, hist["accuracy"], color=c, lw=1.0, alpha=0.85)
        axes[1].plot(it, hist["mean_reward"], color=c, lw=1.0, alpha=0.85)
        axes[2].plot(it, hist["mem_temperature"], color=c, lw=1.0, alpha=0.85)
    axes[0].set(xlabel="episodes (k)", ylabel="recall accuracy",
                title="Performance")
    axes[1].set(xlabel="episodes (k)", ylabel="mean reward / trial",
                title="Reward")
    axes[2].set(xlabel="episodes (k)", ylabel="retrieval temperature τ",
                title="Retrieval sharpening", yscale="log")
    axes[0].legend(handles=[plt.Line2D([], [], color=c, label=k)
                            for k, c in COND_COLORS.items()], fontsize=7)
    fig.tight_layout()
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / "fig_training_population.png", dpi=200,
                bbox_inches="tight")
    plt.show()

---
## Recording protocol and behavioural metrics

### The record protocol

For all analyses the paper switches to a **record model**: identical weights, but (i) the
retrieval is *hard* ($\hat{\mathbf{s}}_t = \mathrm{onehot}(\arg\max s^i_t)$) and
(ii) the temperature sits at its annealed floor $\tau = 5\times10^{-4}$ — together these
make the network retrieve **single memories**, as a human would report single words. For
each model we record and cache three datasets (`record_episodes`):

| recording | trials | actions | purpose |
|---|---|---|---|
| `record_main` | 1 000 | sampled | behaviour (CRP/FA/TS), PCA, variance explained |
| `record_full` | 500 | argmax | cross-decoding, full 64-item pool |
| `record_restricted` | 200 | argmax, 8 fixed items | cross-decoding, restricted pool (item invariance) |

### Behavioural metrics (paper Methods, Eq. 12–13)

**CRP curve.** For every pair of consecutive recalls that both land on studied items, the
transition is binned by its lag $j = i_{t+1} - i_t$ in list positions. Bin counts are divided
by the number of (position, lag) combinations — the uniform-random baseline — and
renormalized:

$$ \mathrm{CRP}(j) \;\propto\; \frac{\mathrm{count}(j)}{\text{\#(position, lag) pairs at } j} .$$

**Forward asymmetry** (Eq. 12):

$$ \mathrm{FA} = \frac{\left|\{\mathrm{RT}(i,j)\,|\, i<j\}\right|}
{\left|\{\mathrm{RT}(i,j)\,|\, i,j\}\right|} \in [0,1], $$

the fraction of valid transitions that go forward; 1 = perfect forward recall, 0 = perfect
backward, 0.5 = no order bias.

**Temporal organization score** (Eq. 13; Polyn et al. 2009). For a transition $\mathrm{RT}(i,j)$
whose set of *possible* continuations is $\mathcal{M}_i$ (not-yet-recalled items), rank all
$\mathrm{RT}(i,k),\, k \in \mathcal{M}_i$ by descending $|k-i|$ (ties share the mean rank)
and score the actual transition by its normalized rank:

$$ \mathrm{TS}(i,j) = \frac{\mathrm{rank}\,\mathrm{RT}(i,j) - 1}{|\mathcal{M}_i| - 1}. $$

The model's TS is the mean over **valid scored transitions** (the reference `TemporalFactorV2`:
intrusions, repeats and the final forced choice with a single possible continuation are
excluded from the mean rather than counted as 0, so perfect forward recall scores exactly
1.0). 0.5 = chance-level contiguity.

In [ ]:
"""Record trained-agent episodes with full hidden-state traces.

Port of ``train/record.py`` (sampled actions, the default) and of the
``experiments/Basic/item_invariant.py`` recorder (argmax actions + forced item
lists, used for cross-decoding), unified into one function.

Per trial the recorder stores:

  * ``study_items``       (8,)    studied item ids (0..63), presentation order
  * ``recall_actions``    (8,)    raw actions 0..64 (0 = no action)
  * ``recall_items``      (8,)    recalled item ids (action - 1; -1 = no action)
  * ``recall_correct``    (8,)    bool mask of correct new recalls
  * ``rewards``           (8,)    response-phase rewards
  * ``study_states``      (8, H)  h after each study step (post-encode)
  * ``recall_cue_states`` (8, H)  h immediately before each retrieval
  * ``recall_out_states`` (8, H)  h that produced each recall (policy input)
  * ``mem_weights``       (8, K)  softmaxed retrieval weights (argmax = slot)
  * ``retrieved_slots``   (8,)    argmax of ``mem_weights``
"""

import numpy as np
import torch


@torch.no_grad()
def record_episodes(agent, env, n_trials: int = 1000, *, mode: str = "sample",
                    device: str = "cpu", seed: int | None = None,
                    reset_options: list | None = None,
                    verbose: bool = False) -> dict:
    """Run ``n_trials`` free-recall trials with a trained agent.

    mode="sample": actions drawn from the policy (reference ``record.py``).
    mode="argmax": greedy actions (reference ``item_invariant.py``), which also
    makes analyses exactly reproducible.

    reset_options: optional per-trial ``reset(options=...)`` dicts, e.g.
    ``{"memory_sequence_index": [...]}`` to force item lists (cross-decoding).
    """
    if mode not in ("sample", "argmax"):
        raise ValueError(f"mode must be 'sample' or 'argmax', got {mode!r}")
    agent.to(device)
    agent.eval()
    core = agent.core
    if seed is not None:
        torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    L = env.sequence_len
    out = {name: [] for name in ("study_items", "recall_actions", "recall_items",
                                 "recall_correct", "rewards", "study_states",
                                 "recall_cue_states", "recall_out_states",
                                 "mem_weights", "retrieved_slots")}

    for trial in range(n_trials):
        opts = reset_options[trial] if reset_options is not None else None
        obs, info = env.reset(options=opts)
        core.reset_memory(1)
        state = core.init_state(1, device)
        core.set_phase(encoding=True)

        study_items, recall_actions, recall_items = [], [], []
        recall_correct, rewards = [], []
        study_states, cue_states, out_states = [], [], []
        mem_weights, retrieved_slots = [], []

        for t in range(2 * L):
            if info.get("reset_state"):
                state = core.init_state(1, device, recall=True, prev_state=state)
            x = torch.as_tensor(np.asarray(obs)[None], dtype=torch.float32,
                                device=device)
            z_prev = state
            state = core.recurrence(x, state)          # retrieve -> GRU -> encode
            logits = agent.readout(state)

            if t < L:
                # study step: the observation carried item t; h is now stored
                study_items.append(int(env.memory_sequence_int[t]))
                study_states.append(state[0].cpu().numpy())
                action = int(torch.distributions.Categorical(logits=logits[0]).sample())
            else:
                # response step: record the cue / output states and retrieval
                cue = core.last_cue if core.last_cue is not None else z_prev
                cue_states.append(cue[0].cpu().numpy())
                out_states.append(state[0].cpu().numpy())
                w = core.last_similarity[0].cpu().numpy()
                mem_weights.append(w)
                retrieved_slots.append(int(np.argmax(w)))
                if mode == "argmax":
                    action = int(torch.argmax(logits[0]))
                else:
                    action = int(torch.distributions.Categorical(logits=logits[0]).sample())

            obs, r, term, trunc, info = env.step(action)
            if t >= L:
                recall_actions.append(action)
                recall_items.append(action - 1)
                recall_correct.append(bool(info.get("correct", 0)))
                rewards.append(float(r))
            phase_recall = info["phase"] == "recall"
            core.set_phase(encoding=not phase_recall, retrieving=phase_recall)

        out["study_items"].append(study_items)
        out["recall_actions"].append(recall_actions)
        out["recall_items"].append(recall_items)
        out["recall_correct"].append(recall_correct)
        out["rewards"].append(rewards)
        out["study_states"].append(study_states)
        out["recall_cue_states"].append(cue_states)
        out["recall_out_states"].append(out_states)
        out["mem_weights"].append(mem_weights)
        out["retrieved_slots"].append(retrieved_slots)
        if verbose and (trial + 1) % 500 == 0:
            print(f"  recorded {trial + 1}/{n_trials}", flush=True)

    arrays = {}
    for name, rows in out.items():
        if name in ("study_items", "recall_actions", "recall_items", "retrieved_slots"):
            arrays[name] = np.array(rows, dtype=int)
        elif "states" in name or name == "mem_weights":
            arrays[name] = np.array(rows, dtype=np.float32)
        else:
            arrays[name] = np.array(rows, dtype=bool if name == "recall_correct" else float)
    arrays["accuracy"] = float(arrays["recall_correct"].mean())
    return arrays


def forced_list_reset_options(rng: np.random.Generator, vocabulary_size: int = 64,
                              n_trials: int = 200, list_len: int = 8,
                              n_unique_items: int = 8) -> list[dict]:
    """Per-trial reset options drawn from a restricted item pool (the paper's
    cross-decoding protocol: train trials contain only ``n_unique_items``)."""
    pool = rng.choice(vocabulary_size, n_unique_items, replace=False)
    return [{"memory_sequence_index": rng.choice(pool, list_len, replace=False)}
            for _ in range(n_trials)]


def evaluate_noise_robustness(agent, env, noise_levels, n_trials: int = 200,
                              mode: str = "argmax", device: str = "cpu",
                              seed: int = 0) -> np.ndarray:
    """Recall accuracy as a function of test-time working-memory noise.

    Paper Fig. 4b: at each noise level the per-step hidden-state noise
    (``wm_noise_prop``) is set to that level and fresh trials are recorded.
    The agent's own training-time noise value is restored afterwards.
    """
    original = agent.core.config.wm_noise_prop
    accuracies = []
    try:
        for level in noise_levels:
            agent.core.config.wm_noise_prop = float(level)
            rec = record_episodes(agent, env, n_trials, mode=mode, device=device,
                                  seed=seed)
            accuracies.append(rec["accuracy"])
    finally:
        agent.core.config.wm_noise_prop = original
    return np.asarray(accuracies)

In [ ]:
"""Free-recall behaviour and representation analyses.

Faithful numpy/sklearn ports of the reference ``analysis/`` package:

  * CRP curve + forward asymmetry   <- analysis/behavior/recall_probability.py
  * temporal organization score     <- analysis/behavior/temporal_factor.py
  * index/identity explained var.   <- analysis/decoding/multi_regression.py
  * per-time-step Ridge decoding    <- analysis/decoding/item_{index,identity}.py
  * cross-phase decoding            <- analysis/decoding/cross_classification.py

Conventions: ``study_items`` (N, 8) holds studied item ids in presentation
order; ``recall_items`` (N, 8) holds recalled ids (-1 = no action).
"""

import numpy as np
from scipy.stats import rankdata
from sklearn.linear_model import LinearRegression, RidgeClassifier
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold


# ---------------------------------------------------------------- behaviour
def recall_transition_matrix(study_items: np.ndarray,
                             recall_items: np.ndarray) -> np.ndarray:
    """Count transitions first(i) -> next(j) between consecutive recalls.

    Reproduces ``RecallProbability.fit``: both items must be in the studied
    list; repeats (i == j) are kept and fall into the lag-0 bin.
    """
    n_trials, L = study_items.shape
    counts = np.zeros((L, L))
    for i in range(n_trials):
        pos = {item: k for k, item in enumerate(study_items[i])}
        rec = recall_items[i]
        for t in range(L - 1):
            if rec[t] in pos and rec[t + 1] in pos:
                counts[pos[rec[t]], pos[rec[t + 1]]] += 1
    return counts


def forward_asymmetry(counts: np.ndarray) -> float:
    """P(forward transition) = upper triangle / total (reference definition)."""
    total = counts.sum()
    return float(np.triu(counts, k=1).sum() / total) if total else np.nan


def crp_curve(study_items: np.ndarray, recall_items: np.ndarray) -> np.ndarray:
    """Conditional-recall probability by lag -L+1 .. L-1.

    Reference normalization: sum the transition matrix by lag, divide by the
    number of (position, lag) combinations (the uniform-random recall baseline)
    and renormalize to a probability distribution.
    """
    counts = recall_transition_matrix(study_items, recall_items)
    _, L = study_items.shape
    by_lag = np.zeros(2 * L - 1)
    for i in range(L):
        by_lag[L - 1 - i: 2 * L - 1 - i] += counts[i]
    average_times = np.concatenate([np.arange(1, L + 1), np.arange(L - 1, 0, -1)])
    by_lag = by_lag / average_times
    return by_lag / by_lag.sum()


def temporal_organization_score(study_items: np.ndarray,
                                recall_items: np.ndarray) -> float:
    """Mean temporal organization score (Polyn et al. 2009 "temporal factor").

    Port of ``TemporalFactor(V2).fit``: each valid transition to the item at
    serial distance |lag| is scored (rank - 1) / (n_possible - 1), where
    possible transitions are ranked by descending |lag| (ties share the mean
    rank). Following the reference V2 variant, the mean is taken over SCORED
    transitions only (invalid transitions and the final forced choice with a
    single possible continuation are excluded, not counted as 0), so perfect
    forward recall scores 1.0.
    """
    n_trials, L = study_items.shape
    scored = []
    for i in range(n_trials):
        study = study_items[i]
        rec = recall_items[i]
        pos = {item: k for k, item in enumerate(study)}
        recalled = np.zeros(L, dtype=bool)
        for t in range(L - 1):
            if rec[t] not in pos or rec[t + 1] not in pos:
                continue
            p1, p2 = pos[rec[t]], pos[rec[t + 1]]
            if recalled[p1] or recalled[p2] or p1 == p2:
                continue
            recalled[p1] = True
            not_yet = ~np.isin(study, rec[:t + 1])
            actual = study[not_yet] == rec[t + 1]
            lags = np.arange(L)[not_yet] - p1
            if len(lags) > 1 and np.any(actual):
                ranks = rankdata(-np.abs(lags))
                scored.append((ranks[actual] - 1).item() / (len(ranks) - 1))
    return float(np.mean(scored)) if scored else float("nan")


# ------------------------------------------------- representational analyses
def explained_variance(states: np.ndarray, index: np.ndarray, identity: np.ndarray,
                       mask: np.ndarray | None = None,
                       n_splits: int = 5, seed: int = 0,
                       lstsq_rcond: float = 1e-10) -> tuple[float, float]:
    """Variance of the hidden states explained by item index vs identity.

    Port of ``MultiRegressor.fit``: joint linear regression of the (flattened)
    states on the one-hot index and one-hot identity; per test fold, the
    regression sum of squares of each predictor block is divided by the total
    sum of squares of the test states. Index and identity are statistically
    independent here, so the two shares do not overlap.

    Numerical note: the one-hot design is exactly collinear with the intercept
    (each row sums to 1), so we solve the least squares with an explicit
    singular-value cutoff ``lstsq_rcond`` (sklearn's default rcond lets the
    near-null directions amplify to ~1e13-magnitude coefficients and explode
    the ratio). On trained models both versions agree; the cutoff only keeps
    the ratio bounded for degenerate (e.g. untrained) models.
    """
    if mask is None:
        mask = np.ones_like(index, dtype=bool)
    X_states = states.reshape(-1, states.shape[-1])
    index = index.reshape(-1)
    identity = identity.reshape(-1)
    mask = mask.reshape(-1).astype(bool)

    onehot_index = np.eye(index.max() + 1)[index]
    onehot_identity = np.eye(identity.max() + 1)[identity]
    design = np.concatenate([onehot_index, onehot_identity], axis=-1)[mask]
    Y = X_states[mask]

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    r2_index, r2_identity = [], []
    n_idx = onehot_index.shape[1]
    for train, test in kf.split(design):
        beta, _, _, _ = np.linalg.lstsq(design[train], Y[train], rcond=lstsq_rcond)
        pred = design[test] @ beta
        ss_total = ((Y[test] - Y[test].mean(0)) ** 2).sum()
        ss_index = ((design[test][:, :n_idx] @ beta[:n_idx]) ** 2).sum()
        ss_identity = ((design[test][:, n_idx:] @ beta[n_idx:]) ** 2).sum()
        r2_index.append(ss_index / ss_total)
        r2_identity.append(ss_identity / ss_total)
    return float(np.mean(r2_index)), float(np.mean(r2_identity))


def decode_identity_matrix(states: np.ndarray, items: np.ndarray,
                           mask: np.ndarray | None = None,
                           n_splits: int = 3, seed: int = 0) -> np.ndarray:
    """Within-phase identity decoding heatmap.

    Port of ``ItemIdentityDecoder.fit``: entry [i, j] decodes the item shown at
    time j from the hidden states at time i (cross-validated over trials), so
    the diagonal is the classic within-time decoding accuracy.
    states: (N, T, H); items: (N, T); mask: (N, T).
    """
    if mask is None:
        mask = np.ones_like(items, dtype=bool)
    n_trials, T, _ = states.shape
    results = np.zeros((T, T))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for i in range(T):
        data_i = states[:, i, :]
        for j in range(T):
            if not mask[:, j].any():
                continue
            X, y = data_i[mask[:, j]], items[:, j][mask[:, j]]
            acc = 0.0
            for train, test in kf.split(X):
                clf = RidgeClassifier().fit(X[train], y[train])
                acc += float((clf.predict(X[test]) == y[test]).mean())
            results[i, j] = acc / n_splits
    return results


def decode_index_by_time(states: np.ndarray, index: np.ndarray,
                         mask: np.ndarray | None = None,
                         n_splits: int = 5, seed: int = 0) -> np.ndarray:
    """Index decoding accuracy per time step (pooled across time for training).

    Port of ``ItemIndexDecoder.fit``: one Ridge classifier trained on all time
    steps, tested separately at each time step.
    """
    if mask is None:
        mask = np.ones_like(index, dtype=bool)
    n_trials, T, _ = states.shape
    X_flat = states.reshape(-1, states.shape[-1])
    y_flat = index.reshape(-1)
    m_flat = mask.reshape(-1).astype(bool)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    accs = np.zeros(T)
    for train, test in kf.split(X_flat):
        clf = RidgeClassifier().fit(X_flat[train][m_flat[train]],
                                    y_flat[train][m_flat[train]])
        for t in range(T):
            sel = np.zeros(n_trials * T, dtype=bool)
            sel[t::T] = mask[:, t]
            sel &= np.isin(np.arange(n_trials * T), test)
            if sel.sum() == 0:
                continue
            accs[t] += float((clf.predict(X_flat[sel]) == y_flat[sel]).mean())
    return accs / n_splits


def _fit_ridge(X, y):
    return RidgeClassifier().fit(X, y)


def cross_decode_index(train_states: np.ndarray, train_index: np.ndarray,
                       test_states: np.ndarray, test_index: np.ndarray,
                       test_mask: np.ndarray | None = None) -> tuple[float, float]:
    """Cross-phase index decoding (memory-palace item/phase invariance test).

    Port of the reference ``CrossClassifier`` usage in ``experiment.py``: train
    one classifier on all of phase A (e.g. study states from restricted-item
    trials), test on phase B (e.g. response states from all-item trials).
    Returns (accuracy, r2).
    """
    clf = _fit_ridge(train_states.reshape(-1, train_states.shape[-1]),
                     train_index.reshape(-1))
    X = test_states.reshape(-1, test_states.shape[-1])
    y = test_index.reshape(-1)
    if test_mask is not None:
        m = test_mask.reshape(-1).astype(bool)
        X, y = X[m], y[m]
    pred = clf.predict(X)
    return float((pred == y).mean()), float(r2_score(y, pred))


def cross_decode_identity(train_states: np.ndarray, train_items: np.ndarray,
                          test_states: np.ndarray, test_items: np.ndarray,
                          test_mask: np.ndarray | None = None,
                          n_train_steps: int = 1, n_repeats: int = 5,
                          seed: int = 0) -> tuple[float, float]:
    """Cross-phase identity decoding from a few training time steps.

    Port of ``time_invariant.py``: the decoder is trained on hidden states from
    ``n_train_steps`` study-phase time step(s) only and tested on every
    response-phase time step (and vice versa by the caller). Averaged over
    ``n_repeats`` random choices of the training step(s).
    """
    rng = np.random.default_rng(seed)
    T_train = train_states.shape[1]
    accs, r2s = [], []
    X_test = test_states.reshape(-1, test_states.shape[-1])
    y_test = test_items.reshape(-1)
    if test_mask is not None:
        m = test_mask.reshape(-1).astype(bool)
        X_test, y_test = X_test[m], y_test[m]
    for _ in range(n_repeats):
        steps = rng.choice(T_train, n_train_steps, replace=False)
        X_train = train_states[:, steps, :].reshape(-1, train_states.shape[-1])
        y_train = train_items[:, steps].reshape(-1)
        clf = _fit_ridge(X_train, y_train)
        pred = clf.predict(X_test)
        accs.append(float((pred == y_test).mean()))
        r2s.append(float(r2_score(y_test, pred)))
    return float(np.mean(accs)), float(np.mean(r2s))


# ------------------------------------------------------------- derived labels
def recall_labels(rec: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Convenience view of a recording for the representational analyses.

    Returns (recall_items, recall_index, valid_mask) where ``recall_index`` is
    the serial position of each recalled item in its list (-1 when invalid).
    """
    recall_items = rec["recall_items"]                       # (N, L), -1 = none
    study_items = rec["study_items"]
    n, L = recall_items.shape
    index = np.full_like(recall_items, -1)
    valid = np.zeros_like(recall_items, dtype=bool)
    for i in range(n):
        pos = {item: k for k, item in enumerate(study_items[i])}
        for t in range(L):
            if recall_items[i, t] in pos:
                index[i, t] = pos[recall_items[i, t]]
                valid[i, t] = True
    return recall_items, index, valid


# ------------------------------------------------------- pooled decoders
def decode_pooled(states: np.ndarray, labels: np.ndarray,
                  mask: np.ndarray | None = None,
                  n_splits: int = 5, seed: int = 0) -> float:
    """Pooled cross-validated decoding accuracy (labels constant per step).

    The scalar ``index_decoding_accuracy`` / ``item_decoding_accuracy`` of the
    reference ``ItemIndexDecoder`` / ``Classifier`` (their
    ``decode_accuracy_all``): one Ridge classifier per fold trained on all time
    steps pooled, accuracy measured on the pooled test set.
    """
    if mask is None:
        mask = np.ones_like(labels, dtype=bool)
    X = states.reshape(-1, states.shape[-1])
    y = labels.reshape(-1)
    m = mask.reshape(-1).astype(bool)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    accs = []
    for train, test in kf.split(X):
        tr = train[m[train]]
        te = test[m[test]]
        clf = RidgeClassifier().fit(X[tr], y[tr])
        accs.append(float((clf.predict(X[te]) == y[te]).mean()))
    return float(np.mean(accs))


def decode_last_item(states: np.ndarray, study_items: np.ndarray,
                     mask: np.ndarray | None = None,
                     n_splits: int = 5, seed: int = 0) -> float:
    """Decode the LAST studied item from the hidden states (recency probe).

    Port of the reference last-item decoder: the label of every time step of a
    trial is the final list item; pooled CV accuracy across steps.
    """
    last = study_items[:, -1:]
    labels = np.broadcast_to(last, (study_items.shape[0], states.shape[1]))
    return decode_pooled(states, labels.copy(), mask=mask,
                         n_splits=n_splits, seed=seed)


# ----------------------------------------------- cross-phase decoding metrics
def cross_decoding_metrics(rec_full: dict, rec_res: dict) -> tuple[float, float]:
    """Cross-condition generalization accuracies (paper Fig. 3b,c protocol).

    index: decoder trained on study states of restricted-item trials, tested on
    response states of full trials (and vice versa), averaged.
    identity: decoder trained on a single study-phase time step, tested on all
    response-phase steps (and vice versa), averaged.
    """
    idx_study = np.tile(np.arange(8), (rec_res["study_states"].shape[0], 1))
    _, rec_index_full, valid_full = recall_labels(rec_full)
    acc_b, _ = cross_decode_index(rec_res["study_states"], idx_study,
                                  rec_full["recall_cue_states"], rec_index_full,
                                  test_mask=valid_full)
    idx_study_full = np.tile(np.arange(8), (rec_full["study_states"].shape[0], 1))
    _, rec_index_res, valid_res = recall_labels(rec_res)
    acc_a, _ = cross_decode_index(rec_full["study_states"], idx_study_full,
                                  rec_res["recall_cue_states"],
                                  rec_index_res, test_mask=valid_res)
    xdec_index = 0.5 * (acc_a + acc_b)

    rec_items_full = np.where(rec_full["recall_items"] < 0, 0,
                              rec_full["recall_items"])
    acc_ir, _ = cross_decode_identity(rec_full["study_states"],
                                      rec_full["study_items"],
                                      rec_full["recall_out_states"],
                                      rec_items_full, test_mask=valid_full)
    acc_ri, _ = cross_decode_identity(rec_full["recall_out_states"],
                                      rec_items_full,
                                      rec_full["study_states"],
                                      rec_full["study_items"], test_mask=None)
    xdec_identity = 0.5 * (acc_ir + acc_ri)
    return xdec_index, xdec_identity


# --------------------------------------------------- per-model metric summary
CLUSTER_FEATURES = ("forward_asymmetry", "temporal_factor",
                    "index_decoding_accuracy", "item_decoding_accuracy",
                    "last_item_decoding_accuracy",
                    "explained_variance_index", "explained_variance_identity",
                    "cross_decoding_accuracy_index",
                    "cross_decoding_accuracy_identity")

STRATEGY_NAMES = {1: "memory_palace", 2: "tcm_forward", 3: "tcm_backward"}


def model_metrics(rec_main: dict, rec_full: dict, rec_res: dict) -> dict:
    """All clustering/figure metrics of one trained model.

    ``rec_main``: sampled-action recording (behaviour, PCA, variance).
    ``rec_full`` / ``rec_res``: argmax recordings on all items / a restricted
    8-item pool (cross-decoding protocol).

    Following the reference ``analysis_build_dataset.ipynb``, every decoding /
    explained-variance feature is computed separately on the study-phase and
    response-phase states and then averaged.
    """
    study = rec_main["study_items"]
    rec_items = rec_main["recall_items"]
    counts = recall_transition_matrix(study, rec_items)

    out = rec_main["recall_out_states"]
    study_states = rec_main["study_states"]
    _, rec_index, valid = recall_labels(rec_main)
    rec_items_safe = np.where(rec_items < 0, 0, rec_items)
    rec_index_safe = np.where(rec_index < 0, 0, rec_index)
    L = study.shape[1]
    study_index = np.tile(np.arange(L), (study.shape[0], 1))

    # explained variance per phase, then averaged (reference protocol)
    r2_idx_enc, r2_id_enc = explained_variance(study_states, study_index, study)
    r2_idx_rec, r2_id_rec = explained_variance(out, rec_index_safe,
                                               rec_items_safe, mask=valid)
    r2_index = 0.5 * (r2_idx_enc + r2_idx_rec)
    r2_identity = 0.5 * (r2_id_enc + r2_id_rec)

    # pooled decoding per phase, then averaged (reference protocol)
    index_acc = 0.5 * (decode_pooled(study_states, study_index)
                       + decode_pooled(out, rec_index_safe, mask=valid))
    item_acc = 0.5 * (decode_pooled(study_states, study)
                      + decode_pooled(out, rec_items_safe, mask=valid))
    last_acc = 0.5 * (decode_last_item(study_states, study)
                      + decode_last_item(out, study, mask=valid))

    xdec_index, xdec_identity = cross_decoding_metrics(rec_full, rec_res)

    return {
        "accuracy": float(rec_main["accuracy"]),
        "forward_asymmetry": forward_asymmetry(counts),
        "temporal_factor": temporal_organization_score(study, rec_items),
        "index_decoding_accuracy": index_acc,
        "item_decoding_accuracy": item_acc,
        "last_item_decoding_accuracy": last_acc,
        "explained_variance_index": r2_index,
        "explained_variance_identity": r2_identity,
        "cross_decoding_accuracy_index": xdec_index,
        "cross_decoding_accuracy_identity": xdec_identity,
        "explained_variance_encoding_index": r2_idx_enc,
        "explained_variance_encoding_identity": r2_id_enc,
        "explained_variance_recall_index": r2_idx_rec,
        "explained_variance_recall_identity": r2_id_rec,
        "crp": crp_curve(study, rec_items),
    }


def cluster_strategies(rows: list[dict], seed: int = 42) -> list[dict]:
    """k-means (k=3) strategy clustering, port of ``analysis_clustering.ipynb``.

    Trained models with accuracy >= 0.6 and temporal factor >= 0.5 are clustered
    on the nine reference features (raw, unscaled, as in the reference).
    Clusters are named by their metric signature: the highest temporal-
    organization cluster is the memory palace; of the remaining two, the one
    with higher forward asymmetry is TCM-like forward, the other TCM-like
    backward. Reservoir models are excluded from the fit and labelled
    "reservoir". Adds "strategy" (and "cluster_ok") to each row in place.
    """
    from sklearn.cluster import KMeans

    for r in rows:
        r.setdefault("strategy", "other")
        r["cluster_ok"] = False
    fit_rows = [r for r in rows
                if not r.get("reservoir", False)
                and r["accuracy"] >= 0.6 and r["temporal_factor"] >= 0.5]
    if len(fit_rows) < 3:
        return rows
    X = np.array([[r[f] for f in CLUSTER_FEATURES] for r in fit_rows])
    km = KMeans(n_clusters=3, random_state=seed, n_init=10).fit(X)
    centers = km.cluster_centers_
    i_ts = CLUSTER_FEATURES.index("temporal_factor")
    i_fa = CLUSTER_FEATURES.index("forward_asymmetry")
    # name the clusters: highest TS -> memory palace; then FA splits fwd/bwd
    order = np.argsort(-centers[:, i_ts])
    palace = order[0]
    rest = sorted(order[1:], key=lambda c: -centers[c, i_fa])
    name_of = {palace: "memory_palace", rest[0]: "tcm_forward",
               rest[1]: "tcm_backward"}
    for r, lab in zip(fit_rows, km.labels_):
        r["strategy"] = name_of[lab]
        r["cluster_ok"] = True
    return rows

---
## Population results: three strategies emerge (paper Fig. 2a–d)

### The clustering protocol

The paper clusters its whole population of trained networks with **k-means (k = 3)** on nine
metrics that capture both behaviour and representations
(`analysis_clustering.ipynb`; models with accuracy < 0.6 or temporal factor < 0.5 are
excluded, reservoir controls are not part of the fit):

| group | features |
|---|---|
| behaviour | `forward_asymmetry`, `temporal_factor` |
| decoding (Ridge, per phase, averaged) | `index_decoding_accuracy`, `item_decoding_accuracy`, `last_item_decoding_accuracy` |
| variance explained (per phase, averaged) | `explained_variance_index`, `explained_variance_identity` |
| cross-phase/item-set generalization | `cross_decoding_accuracy_index`, `cross_decoding_accuracy_identity` |

The decoding features: a pooled Ridge classifier decodes the **item index** (serial
position), the **item identity**, and the **last studied item** (a recency probe) from the
hidden states, computed separately on study-phase and response-phase states and averaged —
the reference `ridge_classifier_stat.pkl` protocol. The variance features come from the
joint linear regression of the hidden states on one-hot index + one-hot identity (Eq. 14–15;
details in Part VIII). Clusters are named by their metric signature: the cluster with the
highest temporal-organization centre is the memory palace; of the two others, the one with
higher forward asymmetry is TCM-like forward, the last is TCM-like backward.

### What to look for

* **t-SNE / scatter panels** — the three clusters should separate in the behaviour
  (FA × TS) plane and in the representation (index × identity) plane, as in Fig. 2b,c.
* **Accuracy bars** — all strategies solve the task; the palace should be at or near the
  top (Fig. 2d), the reservoir clearly below.

In [ ]:
"""Population-level loading and metric caching for the free-recall study.

The notebook trains/loads a small population of networks (see
``fr_tests/train_population.py``) and clusters them into the paper's three
strategies. This module loads the cached checkpoints + recordings from
``models/free_recall/population/<cond>__seed<k>/`` and computes (once, cached
to ``metrics.json``) the nine clustering features of the reference
``analysis_clustering.ipynb`` pipeline.
"""

import json
from pathlib import Path

import numpy as np


STRATEGY_LABELS = {
    "memory_palace": "Strategy 1: memory palace",
    "tcm_forward": "Strategy 2: TCM-like forward",
    "tcm_backward": "Strategy 3: TCM-like backward",
    "reservoir": "Reservoir RNN",
    "other": "unclassified",
}


def load_recording(model_dir: Path, name: str) -> dict:
    """Load a cached recording npz (record_main / record_full / ...)."""
    data = np.load(model_dir / name, allow_pickle=False)
    return {k: data[k] for k in data.files}


def population_dirs(pop_dir: Path) -> list[Path]:
    """All *finished* population members: checkpoint + run meta + the main
    recording (only written after training completes), so a population that
    is still being trained is loaded without the in-progress models."""
    return sorted(d for d in Path(pop_dir).iterdir()
                  if d.is_dir() and (d / "config.json").exists()
                  and (d / "run_meta.json").exists()
                  and (d / "record_main.npz").exists())


def load_population(pop_dir: Path, recompute: bool = False) -> list[dict]:
    """Load all population members and their metrics.

    Returns one row per model: directory meta (name, condition, seed, gamma,
    flush, reservoir) + the clustering features (from ``model_metrics``) +
    noise-robustness curve. Metrics are cached in ``pop_dir/metrics.json``;
    pass ``recompute=True`` to rebuild them from the cached recordings.
    """
    pop_dir = Path(pop_dir)
    cache = pop_dir / "metrics.json"
    rows = []
    for d in population_dirs(pop_dir):
        with open(d / "run_meta.json", encoding="utf-8") as f:
            meta = json.load(f)
        rows.append({"name": d.name, "dir": str(d), **meta})

    cached = {}
    if cache.exists() and not recompute:
        with open(cache, encoding="utf-8") as f:
            cached = json.load(f)

    dirty = False
    for r in rows:
        if r["name"] in cached:
            r.update(cached[r["name"]])
            continue
        rec_main = load_recording(Path(r["dir"]), "record_main.npz")
        rec_full = load_recording(Path(r["dir"]), "record_full.npz")
        rec_res = load_recording(Path(r["dir"]), "record_restricted.npz")
        m = model_metrics(rec_main, rec_full, rec_res)
        m["crp"] = list(np.asarray(m["crp"], dtype=float))
        r.update(m)
        dirty = True
        print(f"[metrics] {r['name']}: acc {m['accuracy']:.3f}, "
              f"FA {m['forward_asymmetry']:.2f}, TS {m['temporal_factor']:.2f}",
              flush=True)

    for r in rows:
        noise = Path(r["dir"]) / "noise.npz"
        if noise.exists():
            data = np.load(noise)
            r["noise_levels"] = [float(x) for x in data["noise_levels"]]
            r["noise_accuracy"] = [float(x) for x in data["accuracy"]]

    rows = cluster_strategies(rows)

    if dirty or not cache.exists():
        serializable = [{k: v for k, v in r.items() if k != "dir"}
                        for r in rows]
        with open(cache, "w", encoding="utf-8") as f:
            json.dump({r["name"]: {k: v for k, v in r.items() if k != "name"}
                       for r in serializable}, f)
    return rows


def strategy_representatives(rows: list[dict]) -> dict[str, dict]:
    """Pick one representative model per strategy (the cluster medoid in
    z-scored behaviour space FA/TS; highest accuracy breaks ties)."""

    reps = {}
    for strat in ("memory_palace", "tcm_forward", "tcm_backward"):
        members = [r for r in rows if r.get("strategy") == strat]
        if not members:
            continue
        X = np.array([[r[f] for f in CLUSTER_FEATURES] for r in members])
        center = X.mean(axis=0)
        scale = X.std(axis=0) + 1e-8
        dist = (((X - center) / scale) ** 2).sum(axis=1)
        reps[strat] = members[int(np.argmin(dist))]
    res = [r for r in rows if r.get("reservoir")]
    if res:
        reps["reservoir"] = max(res, key=lambda r: r["accuracy"])
    return reps

In [ ]:
"""Paper-style figures for the free-recall reproduction.

All figures are saved to ``FIG_DIR`` (``example_supplement/figs/free_recall``)
by the notebook; each function takes plain numpy arrays so it can be called
from the notebook directly. Visual language follows the paper: restrained
palette (strategy colors reused across panels), direct labels, clean spines.
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# strategy palette (reused across every figure; matches the paper's Fig. 2
# legend: teal = memory palace, salmon = TCM-like forward, yellow = TCM-like
# backward, dark navy = reservoir benchmark)
STRATEGY_COLORS = {
    "memory_palace": "#2A9D8F",
    "tcm_forward": "#E76F51",
    "tcm_backward": "#E9C46A",
    "reservoir": "#264653",
    "other": "#BBBBBB",
}
STRATEGY_ORDER = ("memory_palace", "tcm_forward", "tcm_backward", "reservoir")
PHASE_COLORS = ["#4477AA", "#66CCEE", "#228833", "#CCBB44", "#EE6677",
                "#AA3377", "#66CCEE", "#4477AA"]


def setup_matplotlib():
    plt.rcParams.update({
        "figure.dpi": 120, "savefig.dpi": 200, "font.size": 9,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": False, "legend.frameon": False,
        "axes.titlesize": 10, "axes.labelsize": 9,
    })


def _save(fig, fig_dir: Path, name: str):
    fig_dir.mkdir(parents=True, exist_ok=True)
    for ext in ("png",):
        fig.savefig(fig_dir / f"{name}.{ext}", bbox_inches="tight")
    plt.show()
    plt.close(fig)


# ------------------------------------------------------------------ Figure 1
def plot_task_demo(rec: dict, trial: int, fig_dir: Path):
    """One example trial: presented list vs recalled sequence (paper Fig 1a)."""
    setup_matplotlib()
    study = rec["study_items"][trial]
    acts = rec["recall_actions"][trial]
    items = rec["recall_items"][trial]
    correct = rec["recall_correct"][trial]

    fig, axes = plt.subplots(2, 1, figsize=(6.4, 1.9))
    for ax, seq, title in (
            (axes[0], study, "Presented item list"),
            (axes[1], items, "Recalled item list")):
        ax.set_xlim(-0.6, 7.6)
        ax.set_ylim(-0.6, 0.6)
        ax.axis("off")
        ax.set_title(title, loc="left", fontsize=9)
        for k in range(8):
            shown = seq[k]
            if shown >= 0:
                color = "#4477AA" if title.startswith("Presented") else (
                    "#228833" if correct[k] else "#CC3311")
                ax.text(k, 0, str(shown + 1), ha="center", va="center",
                        fontsize=9, color="white",
                        bbox=dict(boxstyle="circle,pad=0.32", fc=color, ec="none"))
            else:
                ax.text(k, 0, "–", ha="center", va="center", fontsize=10,
                        color="#888888")
    _save(fig, fig_dir, "fig1_task_demo")


# ------------------------------------------------------------------ Figure 2
def plot_training_curves(history: dict, fig_dir: Path):
    """Accuracy / reward / retrieval temperature during training."""
    setup_matplotlib()
    it = history["iter"]
    fig, axes = plt.subplots(1, 3, figsize=(9.6, 2.4))
    axes[0].plot(it, history["accuracy"], color="#4477AA")
    axes[0].axhline(1.0, color="#BBBBBB", lw=0.8, ls="--")
    axes[0].set(xlabel="iteration", ylabel="recall accuracy", title="Performance")
    axes[1].plot(it, history["mean_reward"], color="#228833")
    axes[1].axhline(8.0, color="#BBBBBB", lw=0.8, ls="--")
    axes[1].text(0.02, 8.0, " perfect list (8)", fontsize=7, va="bottom",
                 transform=axes[1].get_yaxis_transform(), color="#888888")
    axes[1].set(xlabel="iteration", ylabel="mean reward / trial",
                title="Reward")
    axes[2].plot(it, history["mem_temperature"], color="#EE6677")
    axes[2].set(xlabel="iteration", ylabel="retrieval temperature τ",
                title="Retrieval sharpening")
    for ax in axes:
        ax.set_xlim(0, it[-1])
    fig.tight_layout()
    _save(fig, fig_dir, "fig2_training")


# ------------------------------------------------------------------ Figure 3
def plot_behavior(rec: dict, fig_dir: Path):
    """CRP curve + serial-position of first recall (paper Figs 1d & 2e)."""
    setup_matplotlib()
    study = rec["study_items"]
    rec_items = rec["recall_items"]
    crp = crp_curve(study, rec_items)
    counts = recall_transition_matrix(study, rec_items)
    fa = forward_asymmetry(counts)
    ts = temporal_organization_score(study, rec_items)
    L = study.shape[1]
    lags = np.arange(-(L - 1), L)

    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6))

    ax = axes[0]
    ax.axhline(0, color="#DDDDDD", lw=0.8, zorder=0)
    mask = lags != 0
    ax.plot(lags[mask], crp[mask], "-o", color="#4477AA", ms=4,
            label="conditional recall\nprobability")
    ax.plot([0], crp[~mask], "o", color="#EE6677", ms=5)
    ax.axvline(0, color="#DDDDDD", lw=0.8)
    ax.set(xlabel="lag", ylabel="conditional\nrecall probability",
           title=f"Temporal contiguity  (FA = {fa:.2f}, TS = {ts:.2f})")

    ax = axes[1]
    first = np.array([rec_items[i][np.argmax(rec_items[i] >= 0)]
                      if np.any(rec_items[i] >= 0) else -1
                      for i in range(len(rec_items))])
    study0 = study[:, 0]
    first_pos = np.zeros(L)
    for i in range(len(rec_items)):
        match = np.where(study[i] == first[i])[0]
        if len(match):
            first_pos[match[0]] += 1
    first_pos /= max(len(rec_items), 1)
    ax.bar(np.arange(1, L + 1), first_pos, color="#4477AA")
    ax.set(xlabel="serial position of first recall", ylabel="probability",
           title="Recall initiation")

    fig.tight_layout()
    _save(fig, fig_dir, "fig3_behavior")
    return {"forward_asymmetry": fa, "temporal_organization_score": ts,
            "crp": crp}


# ------------------------------------------------------------------ Figure 4
def plot_pca_trajectories(rec: dict, fig_dir: Path, n_trials: int = 20):
    """Joint PCA of study + response states (paper Fig 2f/h/j)."""
    setup_matplotlib()
    from neuralrnn.analysis import fit_pca

    study = rec["study_states"]          # (N, 8, H)
    cue = rec["recall_cue_states"]       # (N, 8, H)
    n_trials = min(n_trials, study.shape[0])
    X = np.concatenate([study, cue], axis=1)                    # (N, 16, H)
    N, T, H = X.shape
    pca = fit_pca(X.reshape(-1, H), n_components=2)
    proj = pca.transform(X.reshape(-1, H)).reshape(N, T, 2)

    fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.2), sharex=True, sharey=True)
    for t in range(8):
        axes[0].plot(proj[:n_trials, t, 0], proj[:n_trials, t, 1], ".",
                     color=PHASE_COLORS[t], ms=5)
        axes[1].plot(proj[:n_trials, 8 + t, 0], proj[:n_trials, 8 + t, 1], ".",
                     color=PHASE_COLORS[t], ms=5)
    for i in range(n_trials):
        for ax, sl in ((axes[0], slice(0, 8)), (axes[1], slice(8, 16))):
            ax.plot(proj[i, sl, 0], proj[i, sl, 1], "-", color="#CCCCCC",
                    lw=0.6, zorder=0)
    ev = pca.explained_variance_ratio
    axes[0].set(title=f"Study phase (PC1 {ev[0]:.0%}, PC2 {ev[1]:.0%})",
                xlabel="PC1", ylabel="PC2")
    axes[1].set(title="Response phase", xlabel="PC1")
    sm = plt.cm.ScalarMappable(cmap=plt.cm.GnBu_r,
                               norm=plt.Normalize(vmin=1, vmax=8))
    fig.colorbar(sm, ax=axes, shrink=0.85, label="time step", pad=0.02)
    _save(fig, fig_dir, "fig4_pca_trajectories")


def plot_state_similarity(rec: dict, fig_dir: Path):
    """Mean cosine similarity between hidden states within each phase
    (extended-data analysis: smooth drift vs stereotyped scaffold)."""
    setup_matplotlib()
    study = rec["study_states"]
    cue = rec["recall_cue_states"]

    def cosmat(states):
        X = states.reshape(-1, states.shape[-1]).astype(np.float64)
        X = X / np.linalg.norm(X, axis=1, keepdims=True)
        S = X @ X.T
        n = states.shape[0]
        T = states.shape[1]
        S = S.reshape(n, T, n, T).mean(axis=(0, 2))
        return S

    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))
    for ax, states, title in ((axes[0], study, "study phase"),
                              (axes[1], cue, "response phase")):
        im = ax.imshow(cosmat(states), cmap="Blues", vmin=0, vmax=1)
        ax.set(xlabel="time step", ylabel="time step", title=title)
    fig.colorbar(im, ax=axes, shrink=0.85, label="cosine similarity")
    _save(fig, fig_dir, "fig4_state_similarity")


# ------------------------------------------------------------------ Figure 5
def plot_explained_variance(rec: dict, fig_dir: Path) -> dict:
    """Index vs identity variance decomposition (paper Fig 3a / 6e)."""
    setup_matplotlib()

    study = rec["study_states"]
    out = rec["recall_out_states"]
    items = rec["study_items"]
    rec_items, rec_index, valid = recall_labels(rec)
    study_index = np.tile(np.arange(study.shape[1]), (study.shape[0], 1))

    r2 = {}
    r2["index_study"], r2["identity_study"] = explained_variance(
        study, study_index, items)
    r2["index_recall"], r2["identity_recall"] = explained_variance(
        out, rec_index, np.where(rec_items < 0, 0, rec_items), mask=valid)
    # combined (study steps 2-8 + response steps 1-7), as in the reference
    c_all = np.concatenate([study[:, 1:], out[:, :-1]], axis=0)
    index_all = np.concatenate([study_index[:, 1:], rec_index[:, :-1]], axis=0)
    ident_all = np.concatenate([items[:, 1:],
                                np.where(rec_items < 0, 0, rec_items)[:, :-1]], axis=0)
    mask_all = np.concatenate([np.ones_like(items[:, 1:], dtype=bool),
                               valid[:, :-1]], axis=0)
    r2["index_all"], r2["identity_all"] = explained_variance(
        c_all, index_all, ident_all, mask=mask_all)

    fig, ax = plt.subplots(figsize=(4.0, 2.8))
    labels = ["index", "identity"]
    x = np.arange(3)
    vals_i = [r2["index_study"], r2["index_recall"], r2["index_all"]]
    vals_id = [r2["identity_study"], r2["identity_recall"], r2["identity_all"]]
    ax.bar(x - 0.18, vals_i, 0.36, color="#4477AA", label="index")
    ax.bar(x + 0.18, vals_id, 0.36, color="#EE6677", label="identity")
    ax.set_xticks(x, ["study", "response", "combined"])
    ax.set(ylabel="explained variance", title="What the hidden states encode")
    ax.legend()
    fig.tight_layout()
    _save(fig, fig_dir, "fig5_explained_variance")
    return r2


def plot_cross_decoding(rec_restricted: dict, rec_full: dict,
                        fig_dir: Path) -> dict:
    """Cross-condition generalization of index / identity (paper Fig 3b,c)."""
    setup_matplotlib()

    res = {}
    # --- index: train on study states (8-item trials) -> test response states
    idx_study = np.tile(np.arange(8), (rec_restricted["study_states"].shape[0], 1))
    _, rec_index_full, valid_full = recall_labels(rec_full)
    acc_b, _ = cross_decode_index(rec_restricted["study_states"], idx_study,
                                  rec_full["recall_cue_states"], rec_index_full,
                                  test_mask=valid_full)
    idx_study_full = np.tile(np.arange(8), (rec_full["study_states"].shape[0], 1))
    _, rec_index_res, valid_res = recall_labels(rec_restricted)
    acc_a, _ = cross_decode_index(rec_full["study_states"], idx_study_full,
                                  rec_restricted["recall_cue_states"],
                                  rec_index_res, test_mask=valid_res)
    res["index"] = 0.5 * (acc_a + acc_b)

    # --- identity: train on one study time step -> test all response steps
    acc_ir, _ = cross_decode_identity(rec_full["study_states"],
                                      rec_full["study_items"],
                                      rec_full["recall_out_states"],
                                      np.where(rec_full["recall_items"] < 0, 0,
                                               rec_full["recall_items"]),
                                      test_mask=valid_full)
    acc_ri, _ = cross_decode_identity(rec_full["recall_out_states"],
                                      np.where(rec_full["recall_items"] < 0, 0,
                                               rec_full["recall_items"]),
                                      rec_full["study_states"],
                                      rec_full["study_items"],
                                      test_mask=None)
    res["identity"] = 0.5 * (acc_ir + acc_ri)

    fig, ax = plt.subplots(figsize=(3.2, 2.8))
    ax.bar(["index", "identity"], [res["index"], res["identity"]],
           color=["#4477AA", "#EE6677"], width=0.55)
    ax.axhline(1 / 8, color="#BBBBBB", lw=0.8, ls="--")
    ax.text(1.42, 1 / 8, " chance", fontsize=7, color="#888888", va="bottom")
    ax.set(ylabel="cross-phase\ndecoding accuracy", ylim=(0, 1.0),
           title="Generalization across\nphases and item sets")
    fig.tight_layout()
    _save(fig, fig_dir, "fig5_cross_decoding")
    return res


def plot_identity_heatmap(rec: dict, fig_dir: Path):
    """Within-phase identity decoding heatmap (extended-data Fig 2a style)."""
    setup_matplotlib()

    study = rec["study_states"]
    items = rec["study_items"]
    out = rec["recall_out_states"]
    rec_items = np.where(rec["recall_items"] < 0, 0, rec["recall_items"])
    valid = rec["recall_correct"].astype(bool)

    mat_s = decode_identity_matrix(study, items)
    mat_r = decode_identity_matrix(out, rec_items, mask=valid)

    fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.0))
    for ax, mat, title in ((axes[0], mat_s, "study phase"),
                           (axes[1], mat_r, "response phase")):
        im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=1)
        ax.set(xlabel="decoded item (time step)", ylabel="state at time step",
               title=f"identity decoding — {title}")
    fig.colorbar(im, ax=axes, shrink=0.85, label="accuracy")
    _save(fig, fig_dir, "fig5_identity_heatmap")


# ------------------------------------------------------------------ Figure 6
def plot_strategy_metrics(metrics: list[dict], fig_dir: Path):
    """Population view: FA vs TS and index- vs identity-coding, colored by
    working-memory flush (paper Fig 2b,c and Fig 3g lite)."""
    setup_matplotlib()
    metrics = [m for m in metrics if m["accuracy"] >= 0.6]
    fig, axes = plt.subplots(1, 3, figsize=(10.0, 2.9))

    for m in metrics:
        c = STRATEGY_COLORS.get(m.get("strategy"), "#BBBBBB")
        axes[0].scatter(m["forward_asymmetry"], m["temporal_organization_score"],
                        color=c, s=28, zorder=3)
        axes[1].scatter(m["r2_index"], m["r2_identity"], color=c, s=28, zorder=3)
    axes[0].axvline(0.5, color="#DDDDDD", lw=0.8)
    axes[0].set(xlabel="forward asymmetry", ylabel="temporal organization score")
    axes[1].set(xlabel="variance explained (index)",
                ylabel="variance explained (identity)")

    ax = axes[2]
    flushes = sorted({m["flush"] for m in metrics})
    width = 0.8 / max(len(flushes), 1)
    for j, fl in enumerate(flushes):
        sub = [m for m in metrics if m["flush"] == fl]
        n = len(sub)
        palace = sum(m.get("strategy") == "memory_palace" for m in sub) / max(n, 1)
        fwd = sum(m.get("strategy") == "tcm_forward" for m in sub) / max(n, 1)
        bwd = sum(m.get("strategy") == "tcm_backward" for m in sub) / max(n, 1)
        ax.bar(j, palace, width, color=STRATEGY_COLORS["memory_palace"])
        ax.bar(j, fwd, width, bottom=palace, color=STRATEGY_COLORS["tcm_forward"])
        ax.bar(j, bwd, width, bottom=palace + fwd,
               color=STRATEGY_COLORS["tcm_backward"])
    ax.set_xticks(range(len(flushes)), [f"{f:.1f}" for f in flushes])
    ax.set(xlabel="working-memory flush", ylabel="fraction of models",
           title="Strategy vs flush")
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc=STRATEGY_COLORS[k])
                       for k in ("memory_palace", "tcm_forward", "tcm_backward")],
              labels=["memory palace", "TCM-like forward", "TCM-like backward"],
              fontsize=7, loc="upper left", bbox_to_anchor=(1.0, 1.0))

    fig.tight_layout()
    _save(fig, fig_dir, "fig6_strategy_metrics")


def plot_crp_by_flush(metrics: list[dict], fig_dir: Path):
    """Mean CRP curves grouped by flush level (paper Fig 5b style)."""
    setup_matplotlib()
    fig, ax = plt.subplots(figsize=(4.4, 3.0))
    L = None
    for m in sorted(metrics, key=lambda x: -x["flush"]):
        if "crp" not in m:
            continue
        L = (len(m["crp"]) + 1) // 2
        lags = np.arange(-(L - 1), L)
        mask = lags != 0
        ax.plot(lags[mask], m["crp"][mask], "-o", ms=3, lw=1.2,
                color=plt.cm.GnBu_r(0.15 + 0.7 * m["flush"]),
                label=f"flush {m['flush']:.1f}")
    ax.axhline(0, color="#DDDDDD", lw=0.8)
    ax.axvline(0, color="#DDDDDD", lw=0.8)
    ax.set(xlabel="lag", ylabel="conditional recall probability")
    ax.legend(title="flush", fontsize=7)
    fig.tight_layout()
    _save(fig, fig_dir, "fig6_crp_by_flush")


# ------------------------------------------------------------------ Figure 7
def plot_noise_robustness(curves: dict[str, list[float]], noise_levels: np.ndarray,
                          fig_dir: Path):
    """Accuracy under test-time hidden-state noise (paper Fig 4b)."""
    setup_matplotlib()
    fig, ax = plt.subplots(figsize=(4.4, 3.0))
    for name, ys in curves.items():
        ax.plot(noise_levels, ys, "-o", ms=3, lw=1.4, color=STRATEGY_COLORS[name],
                label=name.replace("_", " "))
    ax.set(xlabel="working-memory noise (variance fraction)", ylabel="recall accuracy")
    ax.legend(fontsize=8)
    fig.tight_layout()
    _save(fig, fig_dir, "fig7_noise_robustness")


# ============================================================ population-level
def pca_projection(rec: dict, n_components: int = 2):
    """Joint PCA of study + response(-cue) states of one model.

    Returns (proj, explained_variance_ratio) with proj (N, 16, n_components):
    steps 0-7 = study (post-encode states), 8-15 = response (retrieval cues).
    """
    from neuralrnn.analysis import fit_pca

    study = rec["study_states"]
    cue = rec["recall_cue_states"]
    X = np.concatenate([study, cue], axis=1)                    # (N, 16, H)
    N, T, H = X.shape
    pca = fit_pca(X.reshape(-1, H), n_components=n_components)
    proj = pca.transform(X.reshape(-1, H)).reshape(N, T, n_components)
    return proj, pca.explained_variance_ratio


def plot_crp_on_axis(ax, crp: np.ndarray, color: str, label: str | None = None):
    """One CRP curve on an existing axis (paper Fig. 1d/2e style)."""
    L = (len(crp) + 1) // 2
    lags = np.arange(-(L - 1), L)
    mask = lags != 0
    ax.axhline(0, color="#DDDDDD", lw=0.8, zorder=0)
    ax.plot(lags[mask], np.asarray(crp)[mask], "-o", color=color, ms=3.5,
            lw=1.2, label=label)
    ax.plot([0], np.asarray(crp)[~mask], "o", color="#EE6677", ms=4)
    ax.axvline(0, color="#DDDDDD", lw=0.8)
    ax.set(xlabel="lag", ylabel="conditional\nrecall probability")


def _plot_pca_on_axis(ax, proj: np.ndarray, sl: slice, n_trials: int = 20):
    """PCA trajectory scatter on an existing axis (paper Fig. 2f/h/j/l)."""
    n_trials = min(n_trials, proj.shape[0])
    for i in range(n_trials):
        ax.plot(proj[i, sl, 0], proj[i, sl, 1], "-", color="#CCCCCC",
                lw=0.6, zorder=0)
    for t in range(8):
        ax.plot(proj[:n_trials, sl, 0][:, t], proj[:n_trials, sl, 1][:, t],
                ".", color=PHASE_COLORS[t], ms=4.5)


def plot_population_overview(rows: list[dict], fig_dir: Path):
    """Population clustering overview (paper Fig. 2a-d).

    Panels: t-SNE of the clustering metrics; forward asymmetry vs temporal
    organization; variance explained identity vs index; mean task accuracy per
    strategy (dots = individual models).
    """
    from sklearn.manifold import TSNE

    setup_matplotlib()
    trained = [r for r in rows if r.get("cluster_ok")]
    reservoir = [r for r in rows if r.get("reservoir")]

    fig, axes = plt.subplots(1, 4, figsize=(13.2, 3.0))

    # (a) t-SNE of clustering features (the paper used 405 networks and
    # perplexity 30; with our smaller population perplexity is reduced)
    X = np.array([[r[f] for f in CLUSTER_FEATURES] for r in trained])
    if len(trained) >= 4:
        perp = min(30, max(2, (len(trained) - 1) // 3))
        emb = TSNE(n_components=2, random_state=42,
                   perplexity=perp).fit_transform(X)
        for r, p in zip(trained, emb):
            axes[0].scatter(p[0], p[1], color=STRATEGY_COLORS[r["strategy"]],
                            s=32, alpha=0.85)
    axes[0].set(xlabel="t-SNE component 1", ylabel="t-SNE component 2",
                title="Clustering metrics (t-SNE)")

    # (b) FA vs TS  (c) variance explained index vs identity
    for ax, xf, yf, xl, yl in (
            (axes[1], "forward_asymmetry", "temporal_factor",
             "forward asymmetry", "temporal organization score"),
            (axes[2], "explained_variance_index", "explained_variance_identity",
             "variance explained (index)", "variance explained (identity)")):
        for r in trained:
            ax.scatter(r[xf], r[yf], color=STRATEGY_COLORS[r["strategy"]],
                       s=32, alpha=0.85, zorder=3)
        for r in reservoir:
            ax.scatter(r[xf], r[yf], color=STRATEGY_COLORS["reservoir"],
                       s=32, alpha=0.85, marker="s", zorder=3)
        ax.set(xlabel=xl, ylabel=yl)

    # (d) mean accuracy per strategy
    ax = axes[3]
    groups = [s for s in STRATEGY_ORDER
              if any(r.get("strategy") == s or
                     (s == "reservoir" and r.get("reservoir")) for r in rows)]
    for j, s in enumerate(groups):
        members = ([r for r in rows if r.get("reservoir")] if s == "reservoir"
                   else [r for r in rows if r.get("strategy") == s])
        accs = [r["accuracy"] for r in members]
        ax.bar(j, np.mean(accs), 0.62, color=STRATEGY_COLORS[s],
               yerr=np.std(accs), error_kw=dict(lw=1, capsize=2))
        ax.scatter(np.full(len(accs), j), accs, color="white",
                   edgecolor="#666666", s=18, zorder=3, lw=0.6)
    ax.set_xticks(range(len(groups)),
                  [STRATEGY_LABELS[s].split(":")[0].replace("Strategy ", "S")
                   for s in groups])
    ax.set(ylabel="task accuracy", title="Performance", ylim=(0, 1.05))

    handles = [plt.Rectangle((0, 0), 1, 1, fc=STRATEGY_COLORS[s])
               for s in groups]
    fig.legend(handles, [STRATEGY_LABELS[s] for s in groups],
               loc="lower center", ncol=len(groups), fontsize=8,
               bbox_to_anchor=(0.5, -0.08))
    fig.tight_layout()
    _save(fig, fig_dir, "fig_population_overview")


def plot_strategy_panels(rep_recs: dict[str, dict], fig_dir: Path,
                         n_trials: int = 20):
    """Per-strategy signatures of the representative models (paper Fig. 2e-l).

    ``rep_recs``: strategy name -> its ``record_main`` recording (with a
    ``crp`` entry). One row per strategy (+ reservoir): CRP curve, study-phase
    PCA, response-phase PCA.
    """
    setup_matplotlib()

    groups = [s for s in STRATEGY_ORDER if s in rep_recs]
    fig, axes = plt.subplots(len(groups), 3,
                             figsize=(9.0, 2.35 * len(groups)),
                             squeeze=False, layout="constrained")
    for row, s in enumerate(groups):
        rec = rep_recs[s]
        color = STRATEGY_COLORS[s]
        plot_crp_on_axis(axes[row][0], rec["crp"], color)
        axes[row][0].set_title(STRATEGY_LABELS[s], loc="left", fontsize=9,
                               color=color)
        proj, ev = pca_projection(rec)
        _plot_pca_on_axis(axes[row][1], proj, slice(0, 8), n_trials)
        _plot_pca_on_axis(axes[row][2], proj, slice(8, 16), n_trials)
        axes[row][1].set(xlabel=f"PC1 ({ev[0]:.0%})",
                         ylabel=f"PC2 ({ev[1]:.0%})",
                         title="study phase" if row == 0 else None)
        axes[row][2].set(xlabel=f"PC1 ({ev[0]:.0%})",
                         title="response phase" if row == 0 else None)
    sm = plt.cm.ScalarMappable(cmap=plt.cm.GnBu_r,
                               norm=plt.Normalize(vmin=1, vmax=8))
    fig.colorbar(sm, ax=axes[:, 1:].ravel().tolist(), shrink=0.6,
                 label="time step", pad=0.01)
    _save(fig, fig_dir, "fig_strategy_panels")


def _strategy_groups(rows):
    """(strategy, label, members) in canonical order, reservoir last."""
    groups = []
    for s in STRATEGY_ORDER:
        members = ([r for r in rows if r.get("reservoir")] if s == "reservoir"
                   else [r for r in rows
                         if r.get("strategy") == s and not r.get("reservoir")])
        if members:
            groups.append((s, STRATEGY_LABELS[s], members))
    return groups


def plot_variance_by_strategy(rows: list[dict], fig_dir: Path):
    """Variance explained by item index vs identity per strategy (Fig. 3a)."""
    setup_matplotlib()
    groups = _strategy_groups(rows)
    fig, ax = plt.subplots(figsize=(5.4, 3.0))
    x = np.arange(len(groups))
    w = 0.36
    for j, (s, label, members) in enumerate(groups):
        for off, field, c in ((-w / 2, "explained_variance_index", "#4477AA"),
                              (w / 2, "explained_variance_identity", "#EE6677")):
            vals = [r[field] for r in members]
            color = c
            if s == "reservoir":
                color = "#264653" if field.endswith("index") else "#7a8b99"
            ax.bar(j + off, np.mean(vals), w, color=color,
                   yerr=np.std(vals), error_kw=dict(lw=1, capsize=2))
            ax.scatter(np.full(len(vals), j + off), vals, color="white",
                       edgecolor="#666666", s=16, zorder=3, lw=0.6)
    ax.set_xticks(x, [g[1].replace("Strategy ", "S").replace(": ", ":\n")
                      for g in groups], fontsize=8)
    ax.set(ylabel="variance explained", ylim=0)
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc="#4477AA"),
                       plt.Rectangle((0, 0), 1, 1, fc="#EE6677")],
              labels=["item index", "item identity"], fontsize=8)
    fig.tight_layout()
    _save(fig, fig_dir, "fig_variance_by_strategy")


def plot_cross_decoding_by_strategy(rows: list[dict], fig_dir: Path):
    """Cross-phase/item-set decoding of index and identity per strategy
    (paper Fig. 3b,c)."""
    setup_matplotlib()
    groups = _strategy_groups(rows)
    fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.0))
    for ax, field, title in (
            (axes[0], "cross_decoding_accuracy_index",
             "cross-decoding of item index"),
            (axes[1], "cross_decoding_accuracy_identity",
             "cross-decoding of item identity")):
        for j, (s, label, members) in enumerate(groups):
            vals = [r[field] for r in members]
            ax.bar(j, np.mean(vals), 0.62, color=STRATEGY_COLORS[s],
                   yerr=np.std(vals), error_kw=dict(lw=1, capsize=2))
            ax.scatter(np.full(len(vals), j), vals, color="white",
                       edgecolor="#666666", s=16, zorder=3, lw=0.6)
        ax.axhline(1 / 8, color="#BBBBBB", lw=0.8, ls="--")
        ax.set_xticks(range(len(groups)),
                      [g[1].split(":")[0].replace("Strategy ", "S")
                       for g in groups])
        ax.set(ylabel="decoding accuracy", title=title, ylim=(0, 1.05))
    fig.tight_layout()
    _save(fig, fig_dir, "fig_cross_decoding_by_strategy")


def plot_metric_relations(rows: list[dict], fig_dir: Path):
    """Behaviour <-> representation links (paper Fig. 3d,e): forward asymmetry
    and temporal organization vs variance explained by item index."""
    from scipy.stats import linregress, pearsonr

    setup_matplotlib()
    trained = [r for r in rows if r.get("cluster_ok")]
    reservoir = [r for r in rows if r.get("reservoir")]
    fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.0))
    for ax, xf, xl in ((axes[0], "forward_asymmetry", "forward asymmetry"),
                       (axes[1], "temporal_factor",
                        "temporal organization score")):
        xs = np.array([r[xf] for r in trained])
        ys = np.array([r["explained_variance_index"] for r in trained])
        for r in trained:
            ax.scatter(r[xf], r["explained_variance_index"],
                       color=STRATEGY_COLORS[r["strategy"]], s=30, alpha=0.85,
                       zorder=3)
        for r in reservoir:
            ax.scatter(r[xf], r["explained_variance_index"],
                       color=STRATEGY_COLORS["reservoir"], s=30, marker="s",
                       alpha=0.85, zorder=3)
        if len(trained) >= 3:
            fit = linregress(xs, ys)
            xx = np.linspace(xs.min(), xs.max(), 50)
            ax.plot(xx, fit.intercept + fit.slope * xx, color="#888888",
                    lw=1.0, ls="--")
            r_, p_ = pearsonr(xs, ys)
            ax.set_title(f"Pearson r = {r_:.2f}, P = {p_:.1e}", fontsize=8)
        ax.set(xlabel=xl, ylabel="variance explained (index)")
    fig.tight_layout()
    _save(fig, fig_dir, "fig_metric_relations")


def plot_condition_distribution(rows: list[dict], fig_dir: Path):
    """Strategy distribution per training condition (paper Fig. 3f,g lite)."""
    setup_matplotlib()

    conds = []
    for r in rows:
        if r["condition"] not in conds:
            conds.append(r["condition"])
    fig, ax = plt.subplots(figsize=(4.6, 3.2))
    x = np.arange(len(conds))
    bottoms = np.zeros(len(conds))
    for s in STRATEGY_ORDER[:3]:
        frac = []
        for c in conds:
            members = [r for r in rows if r["condition"] == c
                       and not r.get("reservoir")]
            frac.append(np.mean([r.get("strategy") == s for r in members])
                        if members else 0.0)
        ax.bar(x, frac, 0.62, bottom=bottoms, color=STRATEGY_COLORS[s],
               label=STRATEGY_LABELS[s])
        bottoms += np.array(frac)
    ax.set_xticks(x, conds, fontsize=8)
    ax.set(ylabel="fraction of models", ylim=(0, 1.0),
           xlabel="training condition")
    ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    fig.tight_layout()
    _save(fig, fig_dir, "fig_condition_distribution")


def plot_noise_by_strategy(rows: list[dict], fig_dir: Path):
    """Accuracy under test-time hidden-state noise, mean +/- s.d. across the
    models of each strategy (paper Fig. 4b)."""
    setup_matplotlib()
    fig, ax = plt.subplots(figsize=(4.6, 3.0))
    for s, label, members in _strategy_groups(rows):
        curves = [r["noise_accuracy"] for r in members if "noise_accuracy" in r]
        if not curves:
            continue
        levels = np.asarray(members[0]["noise_levels"])
        Y = np.asarray(curves)
        ax.plot(levels, Y.mean(0), "-o", ms=3, lw=1.4,
                color=STRATEGY_COLORS[s], label=label)
        ax.fill_between(levels, Y.mean(0) - Y.std(0), Y.mean(0) + Y.std(0),
                        color=STRATEGY_COLORS[s], alpha=0.2, lw=0)
    ax.set(xlabel="working-memory noise (variance fraction)",
           ylabel="recall accuracy", ylim=(0, 1.0))
    ax.legend(fontsize=7.5)
    fig.tight_layout()
    _save(fig, fig_dir, "fig_noise_by_strategy")

In [ ]:
# ------------------------------------------- load the population and cluster
rows = load_population(POP_DIR) if pop_dirs else []

if rows:
    print(f"{'model':24s} {'acc':>5s} {'FA':>5s} {'TS':>5s} "
          f"{'R2idx':>6s} {'R2id':>6s} -> strategy")
    for r in rows:
        print(f"{r['name']:24s} {r['accuracy']:5.2f} {r['forward_asymmetry']:5.2f} "
              f"{r['temporal_factor']:5.2f} {r['explained_variance_index']:6.2f} "
              f"{r['explained_variance_identity']:6.2f} -> {r['strategy']}")
    counts = {}
    for r in rows:
        counts[r["strategy"]] = counts.get(r["strategy"], 0) + 1
    print("\ncluster sizes:", counts)
    plot_population_overview(rows, FIG_DIR)
else:
    print("[skip] no trained population found - "
          "run `python fr_tests/train_population.py --all --device cuda` first")

---
## Per-strategy signatures: behaviour and hidden-state geometry (paper Fig. 2e–l)

The paper shows one representative model per strategy. For each cluster we pick the
**medoid** (the member closest to the cluster mean in the nine-metric space) and show:

* **CRP curve** — the memory palace has a sharp peak at **lag +1** and almost nothing else
  (strict forward serial recall); TCM-like strategies have a **smooth, wide** contiguity
  curve, skewed forward (strategy 2) or backward (strategy 3).
* **Hidden-state PCA** — fit jointly on study states (after each stored item) and response
  states (the retrieval cues). The palace follows a **stereotyped, item-independent
  trajectory** — all trials collapse onto one scaffold — while TCM-like networks drift
  **item-dependently**, so their trajectories splay apart. The reservoir, with random
  frozen dynamics, is item-driven by construction.

In [ ]:
# ------------------------------- representative model of each strategy
rep_recs = {}
if rows:
    reps = strategy_representatives(rows)
    for strat, row in reps.items():
        rec = load_recording(Path(row["dir"]), "record_main.npz")
        rec["crp"] = row["crp"]
        rep_recs[strat] = rec
        print(f"{strat:15s} -> {row['name']} "
              f"(acc {row['accuracy']:.3f}, FA {row['forward_asymmetry']:.2f}, "
              f"TS {row['temporal_factor']:.2f})")
    plot_strategy_panels(rep_recs, FIG_DIR)
else:
    print("[skip] no trained population found")

In [ ]:
# one example trial of the memory-palace representative (paper Fig. 1a)
if "memory_palace" in rep_recs:
    plot_task_demo(rep_recs["memory_palace"], trial=0, fig_dir=FIG_DIR)

---
## Representations per strategy: index code vs identity code (paper Fig. 3a–e)

### Variance explained by index vs identity (Eq. 14–15)

With $\mathbf{h}$ the hidden state, $x^{\mathrm{index}}$ the one-hot serial position and
$x^{\mathrm{identity}}$ the one-hot item identity,

$$ \mathbf{h} = \beta_0 + \boldsymbol{\beta}_{\mathrm{index}} x^{\mathrm{index}}
+ \boldsymbol{\beta}_{\mathrm{identity}} x^{\mathrm{identity}} + \boldsymbol{\epsilon},
\qquad
R^2_{\mathrm{index}} = \frac{\mathrm{SS}_{\mathrm{index}}}{\mathrm{SS}_{\mathrm{total}}},
\quad
R^2_{\mathrm{identity}} = \frac{\mathrm{SS}_{\mathrm{identity}}}{\mathrm{SS}_{\mathrm{total}}} . $$

Because list composition is random, index and identity are statistically independent and
the two shares do not overlap. The regression is run per phase (study: index = presentation
order; response: index = the recalled item's position in its list) and averaged.

### Cross-phase, cross-item-set decoding (Fig. 3b,c)

The memory-palace index code should be *item-invariant* and *phase-invariant* — the same
"location" scaffold serves encoding and retrieval of any item. We train an index decoder on
study states from trials containing **only 8 fixed items** and test it on response states
from **all 64 items** (and the reverse), averaging the two. For identity, we train on a
**single study time step** and test across **all response time steps** (and vice versa).

### Behaviour ↔ representation correlations (Fig. 3d,e)

Across the whole population, the paper finds that the variance explained by item index
correlates **positively with forward asymmetry** (r = 0.87) and **positively with the
temporal organization score** (r = 0.87): the stronger the item-independent scaffold, the
more systematic the forward recall.

In [ ]:
# ------------------------- variance explained + cross-decoding per strategy
if rows:
    plot_variance_by_strategy(rows, FIG_DIR)
    plot_cross_decoding_by_strategy(rows, FIG_DIR)
else:
    print("[skip] no trained population found")

In [ ]:
# ------------------------- behaviour <-> representation correlations
if rows:
    plot_metric_relations(rows, FIG_DIR)
else:
    print("[skip] no trained population found")

---
## Strategy selection and robustness (paper Fig. 3f,g & 4a,b)

**Which condition produces which strategy?** The stacked-bar summary shows the fraction of
models per training condition assigned to each cluster — our 4-seeds-per-condition analogue
of the paper's full sweeps over discount factor (Fig. 3f) and flush (Fig. 3g).

**Noise robustness.** A truly optimal strategy must survive neural variability (Fig. 4b). We
inject variance-preserving noise into the hidden state at **every step at test time** — the
same $\mathbf{h} \leftarrow \sqrt{1-p}\,\mathbf{h} + \sqrt{p}\,\boldsymbol{\epsilon}\,
\mathrm{std}(\mathbf{h})$ used during training — and measure recall accuracy as a function
of the noise fraction $p$, averaged over the members of each strategy (curves cached during
recording as `noise.npz`).

The palace's index code is a **stable scaffold**: even if individual states are perturbed,
the retrieval cue still lands near the correct "location". A drifting context cue, in
contrast, is deflected by noise and retrieves the wrong memory.

In [ ]:
# ----------------- strategy distribution per condition + noise robustness
if rows:
    plot_condition_distribution(rows, FIG_DIR)
    plot_noise_by_strategy(rows, FIG_DIR)
else:
    print("[skip] no trained population found")

---
## Summary

because the memory module stores *states* and retrieval is
*state similarity*, the cheapest way to recall a whole list in order is to make the
hidden state encode a position-like sequence that is (i) independent of *which* items were
shown and (ii) reinstated identically at recall — exactly the memory palace — and this
solution wins when reward discounting is weak and recency cues are flushed away.


### References

1. Li, M., Jensen, K. T., Zhang, Q., Lu, Q. & Mattar, M. G. (2026). A neural network model of
   free recall learns multiple memory strategies. *Nature Machine Intelligence* 8, 1238–1250.
2. Howard, M. W. & Kahana, M. J. (2002). A distributed representation of temporal context.
   *J. Math. Psychol.* 46, 269–299.
3. Sederberg, P. B., Howard, M. W. & Kahana, M. J. (2008). A context-based theory of recency
   and contiguity in free recall. *Psychol. Rev.* 115, 893.
4. Polyn, S. M., Norman, K. A. & Kahana, M. J. (2009). A context maintenance and retrieval
   model. *Psychol. Rev.* 116, 129.
5. Zhang, Q., Griffiths, T. L. & Norman, K. A. (2023). Optimal policies for free recall.
   *Psychol. Rev.* 130, 1104.
6. Lu, Q., Hasson, U. & Norman, K. A. (2022). A neural network model of when to retrieve and
   encode episodic memories. *eLife* 11, e74445.